# DERM-Net Unified — Rare Skin Disease Classification + Explainable AI

**One notebook. Press `Run All`.** Nothing to edit, no paths to fix, no checkpoints to upload.

This file merges and repairs three earlier notebooks (`Capstone_B_DERMNet_Dual`,
`dermnetonlyimbalanced`, `xaifordermnet_1`) into a single reproducible pipeline.

### What it does
| Stage | Contents |
|---|---|
| 1 | Auto-discovers the dataset anywhere under `/kaggle/input` — no hardcoded paths |
| 2 | EDA, one **canonical** stratified 70/15/15 split reused by *every* stage |
| 3 | DERM-Net (EfficientNet-B4 + ViT-B/16, MSCA fusion, Hybrid Focal+CB-CE+SupCon loss) |
| 4 | Ablation grid over batch size x epochs |
| 5 | Baseline CNN/Transformer comparison |
| 6 | Full metrics: bootstrap 95% CI, AUC-ROC/PR, MCC, kappa, Brier, ECE, calibration |
| 7 | Robustness stress-test (noise / blur / brightness / JPEG / occlusion) |
| 8 | **XAI**: Grad-CAM++, EigenCAM, LayerCAM, ViT Attention Rollout, **LIME**, Occlusion |
| 9 | **Quantitative XAI faithfulness**: deletion / insertion AUC ranking of every method |

### Accuracy improvements over the source notebooks
* **EMA weight averaging** + warmup-cosine schedule + best-on-val-macro-F1 checkpointing
* **Test-time augmentation** (4-view) at inference
* **Temperature scaling** calibrated on the validation split (large Brier/ECE gain)
* **Square-root class-balanced sampling** instead of full inverse-frequency — the original
  combined full oversampling *and* full class-weighted loss, double-correcting the imbalance
  and over-predicting minority classes
* Bootstrap confidence intervals instead of the normal-approximation interval

### Bugs fixed from the originals
* Two hard `SyntaxError`s in the XAI notebook (unclosed parentheses)
* The XAI notebook rebuilt a **different** test split than the checkpoint was trained on —
  silently leaking training images into "test" explanations. One split is now built once.
* Hardcoded `/kaggle/input/datasets/roshni2404/...` paths that do not exist on Kaggle
* Class list hardcoded with a typo (`Ichtyosis`) that silently dropped a class if the folder
  name differed — class names are now discovered from disk
* `torch.cuda.amp` deprecation warnings, single-row `plt.subplots` indexing crashes,
  `num_workers` worker-shutdown warnings, `torch.load` weights-only breakage on torch>=2.6

In [ ]:
# =============================================================================
#  CELL 1 - Dependencies
# =============================================================================
import importlib
import subprocess
import sys

_REQUIRED = [
    ("timm", "timm"),
    ("pytorch_grad_cam", "grad-cam"),
    ("lime", "lime"),
    ("skimage", "scikit-image"),
    ("cv2", "opencv-python-headless"),
    ("tqdm", "tqdm"),
]

_missing = []
for _module, _package in _REQUIRED:
    try:
        importlib.import_module(_module)
    except Exception:
        _missing.append(_package)

if _missing:
    print("Installing:", ", ".join(_missing))
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *_missing],
        check=False,
    )
    importlib.invalidate_caches()
else:
    print("All dependencies already available.")

print("Python", sys.version.split()[0])

# Pretrained ImageNet weights are downloaded from huggingface.co on first use.
# On Kaggle that requires Internet to be switched on for the notebook.
try:
    import urllib.request

    urllib.request.urlopen("https://huggingface.co", timeout=10)
    print("Internet reachable - pretrained backbone weights can be downloaded.")
except Exception:
    print("=" * 78)
    print("  WARNING: huggingface.co is not reachable.")
    print("  Pretrained ImageNet weights cannot be downloaded, and training from")
    print("  random initialisation on a few hundred images will not work.")
    print()
    print("  On Kaggle: right sidebar -> Notebook options -> turn ON 'Internet',")
    print("  then Run All again. (Internet requires a phone-verified account.)")
    print("=" * 78)

In [ ]:
# =============================================================================
#  CELL 2 - Imports, reproducibility, global configuration
# =============================================================================
import copy
import gc
import json
import math
import os
import random
import shutil
import time
import warnings
from collections import Counter, OrderedDict
from pathlib import Path

warnings.filterwarnings("ignore")

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from tqdm.auto import tqdm

import timm

sns.set_style("darkgrid")
plt.rcParams["figure.max_open_warning"] = 0

# np.trapz was removed in NumPy 2.0 and renamed to np.trapezoid. Kaggle images ship
# both major versions depending on when the image was built, so bind it once here.
TRAPZ = getattr(np, "trapezoid", None) or np.trapz


def set_seed(seed: int = 42) -> None:
    """Make every stochastic component in the pipeline reproducible."""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class CFG:
    """Every knob in the pipeline. Nothing else needs editing."""

    SEED = 42
    IMG_SIZE = 224

    # ---- runtime budget -----------------------------------------------------
    # "full" reproduces the paper numbers (~3-5 h on a Kaggle P100/T4).
    # "fast" is a ~35 min smoke test that exercises every single code path.
    RUN_MODE = "full"

    # ---- DERM-Net headline run ---------------------------------------------
    MAIN_BATCH_SIZE = 32
    MAIN_EPOCHS = 40
    BASE_LR = 1e-4          # scaled linearly with batch size
    WEIGHT_DECAY = 1e-4
    WARMUP_EPOCHS = 3
    LABEL_SMOOTHING = 0.1
    DROPOUT = 0.4

    # hybrid loss weights: L = ALPHA*Focal + BETA*CB-CE + GAMMA*SupCon
    ALPHA, BETA, GAMMA = 0.5, 0.3, 0.2
    FOCAL_GAMMA = 2.0

    # imbalance handling. 0.0 = no resampling, 1.0 = full inverse frequency.
    # 0.5 (square-root sampling) pairs correctly with a class-weighted loss.
    SAMPLER_POWER = 0.5

    # ---- accuracy boosters --------------------------------------------------
    USE_AMP = True
    USE_EMA = True
    EMA_DECAY = 0.999
    USE_TTA = True          # 4-view test-time augmentation
    CALIBRATE = True        # temperature scaling fitted on the validation split

    # ---- study design -------------------------------------------------------
    RUN_ABLATION = True
    ABLATION_GRID = [(16, 20), (16, 40), (32, 20), (32, 40)]

    RUN_BASELINES = True
    BASELINE_EPOCHS = 25
    BASELINE_PATIENCE = 7
    BASELINES = [
        ("EfficientNet-B4", "efficientnet_b4", 1e-4),
        ("EfficientNetV2-S", "tf_efficientnetv2_s", 1e-4),
        ("ResNet-50V2", "resnetv2_50", 1e-4),
        ("ResNet-18", "resnet18", 1e-4),
        ("DenseNet-121", "densenet121", 1e-4),
        ("MobileNetV3-L", "mobilenetv3_large_100", 1e-4),
        ("ConvNeXt-Tiny", "convnext_tiny", 5e-5),
        ("ViT-B/16", "vit_base_patch16_224", 5e-5),
        ("DeiT-Small", "deit_small_patch16_224", 5e-5),
        ("Swin-Tiny", "swin_tiny_patch4_window7_224", 5e-5),
    ]

    RUN_ROBUSTNESS = True

    # ---- XAI ----------------------------------------------------------------
    RUN_XAI = True
    LIME_SAMPLES = 1000
    LIME_SEGMENTS = 80
    XAI_FAITHFULNESS_IMAGES = 12   # images used for deletion/insertion scoring
    XAI_FAITHFULNESS_STEPS = 25

    # ---- split --------------------------------------------------------------
    TEST_SIZE = 0.15
    VAL_SIZE = 0.15

    NUM_WORKERS = 2


if CFG.RUN_MODE == "fast":
    CFG.MAIN_EPOCHS = 4
    CFG.WARMUP_EPOCHS = 1
    CFG.ABLATION_GRID = [(32, 3)]
    CFG.BASELINE_EPOCHS = 3
    CFG.BASELINE_PATIENCE = 3
    CFG.BASELINES = CFG.BASELINES[:3]
    CFG.LIME_SAMPLES = 300
    CFG.XAI_FAITHFULNESS_IMAGES = 4

set_seed(CFG.SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = CFG.USE_AMP and DEVICE.type == "cuda"

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./outputs")
FIG_DIR = OUT_DIR / "figures"
CKPT_DIR = OUT_DIR / "checkpoints"
for _d in (OUT_DIR, FIG_DIR, CKPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)


def savefig(fig, name: str) -> None:
    """Save a figure to the figures folder at publication resolution."""
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
    print(f"   saved -> {path}")


IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
MEAN_T = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
STD_T = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)

def autocast_ctx(enabled: bool):
    """torch.amp (>=2.3) with a fallback to the older torch.cuda.amp namespace."""
    try:
        return torch.amp.autocast("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.autocast(enabled=enabled)


def make_grad_scaler(enabled: bool):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


print(f"Device        : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch       : {torch.__version__}   |   timm {timm.__version__}")
print(f"Run mode      : {CFG.RUN_MODE}")
print(f"Output folder : {OUT_DIR}")

## 1. Dataset discovery and exploratory analysis

The source notebooks hardcoded `/kaggle/input/datasets/roshni2404/rare-skin-disease-dataset/Data`,
which is not the path Kaggle actually mounts a dataset at. Instead of guessing, the cell below
walks `/kaggle/input` and scores every directory by "how much does this look like an
`ImageFolder` root" — so it works no matter what the dataset was named when it was attached,
and it works locally too.

In [ ]:
# =============================================================================
#  CELL 3 - Locate the dataset and discover the classes
# =============================================================================
IMG_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff")

# Cosmetic display names. Folder names are matched case/punctuation-insensitively;
# anything unrecognised keeps its folder name, so a new class never breaks the run.
PRETTY_NAMES = {
    "epidermolysisbullosa": "Epidermolysis Bullosa",
    "eb": "Epidermolysis Bullosa",
    "ichtyosis": "Ichthyosis",
    "ichthyosis": "Ichthyosis",
    "hemangiomas": "Hemangiomas",
    "hemangioma": "Hemangiomas",
    "portwine": "Port-Wine Stain",
    "portwinestain": "Port-Wine Stain",
    "pws": "Port-Wine Stain",
    "healthyskin": "Healthy Skin",
    "healthy": "Healthy Skin",
    "normal": "Healthy Skin",
}


def _norm_key(name: str) -> str:
    return "".join(ch for ch in name.lower() if ch.isalnum())


def _count_images(directory: Path) -> int:
    try:
        return sum(1 for f in directory.iterdir() if f.is_file() and f.suffix.lower() in IMG_EXTS)
    except OSError:
        return 0


def find_dataset_root(search_bases=("/kaggle/input", "./data", "."), max_depth: int = 6):
    """Return the directory that best resembles an ImageFolder root.

    A candidate scores by (number of class subfolders, total images). Directories
    named train/val/test are skipped as roots so we always land on the *parent*
    that holds the class folders.
    """
    candidates = []
    for base in search_bases:
        base_path = Path(base)
        if not base_path.is_dir():
            continue
        base_depth = len(base_path.parts)
        for root, dirs, _files in os.walk(base_path):
            root_path = Path(root)
            if len(root_path.parts) - base_depth > max_depth:
                dirs[:] = []
                continue
            dirs[:] = [d for d in dirs if not d.startswith(".")]
            class_dirs = [root_path / d for d in dirs]
            counts = [(d, _count_images(d)) for d in class_dirs]
            counts = [(d, c) for d, c in counts if c >= 5]
            if len(counts) >= 2:
                names = {_norm_key(d.name) for d, _ in counts}
                if names & {"train", "val", "test", "valid", "training", "testing"}:
                    continue
                total = sum(c for _, c in counts)
                candidates.append((len(counts), total, root_path))
    if not candidates:
        return None
    candidates.sort(key=lambda t: (t[0], t[1]), reverse=True)
    return candidates[0][2]


DATA_ROOT = find_dataset_root()
if DATA_ROOT is None:
    raise FileNotFoundError(
        "No image-classification folder found.\n"
        "On Kaggle: 'Add Input' -> search 'Rare Skin Disease Dataset' -> Add, then re-run.\n"
        "Locally: place the class folders under ./data/<class>/*.jpg"
    )

CLASS_DIRS = sorted(
    [d for d in DATA_ROOT.iterdir() if d.is_dir() and _count_images(d) >= 5],
    key=lambda d: d.name.lower(),
)
CLASS_FOLDERS = [d.name for d in CLASS_DIRS]
CLASS_NAMES = [PRETTY_NAMES.get(_norm_key(name), name.replace("_", " ").title()) for name in CLASS_FOLDERS]
NUM_CLASSES = len(CLASS_NAMES)

FILE_PATHS, FILE_LABELS = [], []
for label, class_dir in enumerate(CLASS_DIRS):
    for f in sorted(class_dir.iterdir()):
        if f.is_file() and f.suffix.lower() in IMG_EXTS:
            FILE_PATHS.append(str(f))
            FILE_LABELS.append(label)

FILE_LABELS = np.asarray(FILE_LABELS)
class_counts = Counter(FILE_LABELS.tolist())

print(f"Dataset root : {DATA_ROOT}")
print(f"Classes      : {NUM_CLASSES}")
print(f"Total images : {len(FILE_PATHS)}\n")
print(f"{'#':<3} {'Folder':<28} {'Display name':<24} {'Images':>7} {'Share':>8}")
print("-" * 74)
for i, (folder, pretty) in enumerate(zip(CLASS_FOLDERS, CLASS_NAMES)):
    n = class_counts[i]
    print(f"{i:<3} {folder:<28} {pretty:<24} {n:>7} {100 * n / len(FILE_PATHS):>7.2f}%")
print("-" * 74)
_imb = max(class_counts.values()) / max(1, min(class_counts.values()))
print(f"{'':<32} {'TOTAL':<24} {len(FILE_PATHS):>7}")
print(f"\nImbalance ratio (largest : smallest) = {_imb:.2f} : 1")

In [ ]:
# =============================================================================
#  CELL 4 - EDA figures: class balance and image-property distributions
# =============================================================================
_sample_idx = np.random.RandomState(CFG.SEED).choice(
    len(FILE_PATHS), size=min(300, len(FILE_PATHS)), replace=False
)
_widths, _heights, _brightness = [], [], []
for i in _sample_idx:
    try:
        with Image.open(FILE_PATHS[i]) as im:
            im = im.convert("RGB")
            _widths.append(im.width)
            _heights.append(im.height)
            _brightness.append(float(np.asarray(im.resize((64, 64))).mean()))
    except Exception:
        continue

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Exploratory Data Analysis", fontsize=16, fontweight="bold")

counts = [class_counts[i] for i in range(NUM_CLASSES)]
palette = sns.color_palette("viridis", NUM_CLASSES)
bars = axes[0, 0].bar(range(NUM_CLASSES), counts, color=palette)
axes[0, 0].set_xticks(range(NUM_CLASSES))
axes[0, 0].set_xticklabels(CLASS_NAMES, rotation=30, ha="right", fontsize=9)
axes[0, 0].set_ylabel("Images")
axes[0, 0].set_title(f"Class distribution (imbalance {_imb:.2f}:1)", fontweight="bold")
for bar, value in zip(bars, counts):
    axes[0, 0].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + max(counts) * 0.01,
        str(value), ha="center", fontsize=9, fontweight="bold",
    )

axes[0, 1].pie(counts, labels=CLASS_NAMES, autopct="%1.1f%%", colors=palette,
               textprops={"fontsize": 9})
axes[0, 1].set_title("Class share", fontweight="bold")

axes[1, 0].scatter(_widths, _heights, alpha=0.45, s=18, color="#2196F3")
axes[1, 0].set_xlabel("Width (px)")
axes[1, 0].set_ylabel("Height (px)")
axes[1, 0].set_title(f"Native resolution (n={len(_widths)} sampled)", fontweight="bold")
axes[1, 0].axvline(CFG.IMG_SIZE, ls="--", color="red", lw=1)
axes[1, 0].axhline(CFG.IMG_SIZE, ls="--", color="red", lw=1,
                   label=f"model input {CFG.IMG_SIZE}px")
axes[1, 0].legend(fontsize=8)

axes[1, 1].hist(_brightness, bins=30, color="#FF9800", edgecolor="white")
axes[1, 1].set_xlabel("Mean pixel intensity (0-255)")
axes[1, 1].set_ylabel("Images")
axes[1, 1].set_title("Brightness distribution", fontweight="bold")

plt.tight_layout()
savefig(fig, "01_eda_overview")
plt.show()

In [ ]:
# =============================================================================
#  CELL 5 - Sample images, four per class
# =============================================================================
_rng = np.random.RandomState(CFG.SEED)
n_show = 4
fig, axes = plt.subplots(NUM_CLASSES, n_show, figsize=(3.1 * n_show, 3.1 * NUM_CLASSES))
axes = np.atleast_2d(axes)
if axes.shape[1] == 1:
    axes = axes.reshape(NUM_CLASSES, -1)
fig.suptitle("Representative images per class", fontsize=15, fontweight="bold", y=1.001)

for row in range(NUM_CLASSES):
    idx_pool = np.flatnonzero(FILE_LABELS == row)
    picks = _rng.choice(idx_pool, size=min(n_show, len(idx_pool)), replace=False)
    for col in range(n_show):
        ax = axes[row, col]
        ax.set_xticks([])
        ax.set_yticks([])
        if col < len(picks):
            with Image.open(FILE_PATHS[picks[col]]) as im:
                ax.imshow(im.convert("RGB"))
        else:
            ax.axis("off")
        if col == 0:
            ax.set_ylabel(CLASS_NAMES[row].replace(" ", "\n"), fontsize=10,
                          fontweight="bold", rotation=0, labelpad=52,
                          ha="right", va="center")

plt.tight_layout()
savefig(fig, "02_sample_images")
plt.show()

## 2. The canonical split

**This is the single most important correctness fix in this notebook.** The three source
notebooks built three different splits (one copied files to `/kaggle/working/skin_split` at
75/10/15, one split file paths at 70/15/15, one re-split inside the XAI notebook), and the XAI
notebook then explained a "test" set that overlapped the checkpoint's training data.

Here the split is computed **once**, stratified by class, and every subsequent stage —
ablation, baselines, robustness, XAI — reuses exactly these indices.

In [ ]:
# =============================================================================
#  CELL 6 - One stratified split, reused everywhere
# =============================================================================
_indices = np.arange(len(FILE_PATHS))

train_idx, temp_idx = train_test_split(
    _indices,
    test_size=CFG.TEST_SIZE + CFG.VAL_SIZE,
    stratify=FILE_LABELS,
    random_state=CFG.SEED,
)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=CFG.TEST_SIZE / (CFG.TEST_SIZE + CFG.VAL_SIZE),
    stratify=FILE_LABELS[temp_idx],
    random_state=CFG.SEED,
)

SPLIT = {"train": np.sort(train_idx), "val": np.sort(val_idx), "test": np.sort(test_idx)}

# Integrity assertions - a leak here would invalidate every number downstream.
assert len(set(SPLIT["train"]) & set(SPLIT["val"])) == 0
assert len(set(SPLIT["train"]) & set(SPLIT["test"])) == 0
assert len(set(SPLIT["val"]) & set(SPLIT["test"])) == 0
assert len(SPLIT["train"]) + len(SPLIT["val"]) + len(SPLIT["test"]) == len(FILE_PATHS)

y_train = FILE_LABELS[SPLIT["train"]]
y_val = FILE_LABELS[SPLIT["val"]]
y_test = FILE_LABELS[SPLIT["test"]]

print("Canonical stratified split (disjointness verified)\n")
header = f"{'Class':<24} {'Train':>7} {'Val':>7} {'Test':>7} {'Total':>7}"
print(header)
print("-" * len(header))
for c in range(NUM_CLASSES):
    print(
        f"{CLASS_NAMES[c]:<24} {int((y_train == c).sum()):>7} "
        f"{int((y_val == c).sum()):>7} {int((y_test == c).sum()):>7} "
        f"{class_counts[c]:>7}"
    )
print("-" * len(header))
print(f"{'TOTAL':<24} {len(y_train):>7} {len(y_val):>7} {len(y_test):>7} {len(FILE_PATHS):>7}")

CLASS_WEIGHTS = torch.tensor(
    compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_train),
    dtype=torch.float32,
)
print(f"\nClass weights for the loss : {np.round(CLASS_WEIGHTS.numpy(), 3)}")

In [ ]:
# =============================================================================
#  CELL 7 - Datasets, augmentation, and loader construction
# =============================================================================
class SkinDataset(Dataset):
    """Indexes into the global FILE_PATHS list through a split index array."""

    def __init__(self, indices, transform=None):
        self.indices = np.asarray(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        gid = int(self.indices[i])
        with Image.open(FILE_PATHS[gid]) as im:
            img = im.convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, int(FILE_LABELS[gid])


TRAIN_TF = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    transforms.TrivialAugmentWide(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN.tolist(), IMAGENET_STD.tolist()),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.10), value="random"),
])

EVAL_TF = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN.tolist(), IMAGENET_STD.tolist()),
])


def make_loaders(batch_size: int, sampler_power: float = CFG.SAMPLER_POWER, augment: bool = True):
    """Build train/val/test loaders from the canonical split.

    sampler_power controls imbalance resampling: 0 = natural frequency,
    0.5 = square-root balanced (default, pairs with a class-weighted loss),
    1.0 = full inverse frequency (double-corrects and is usually worse).
    """
    train_ds = SkinDataset(SPLIT["train"], TRAIN_TF if augment else EVAL_TF)
    val_ds = SkinDataset(SPLIT["val"], EVAL_TF)
    test_ds = SkinDataset(SPLIT["test"], EVAL_TF)

    if sampler_power > 0:
        counts = np.bincount(y_train, minlength=NUM_CLASSES).astype(np.float64)
        per_class_w = (1.0 / np.maximum(counts, 1)) ** sampler_power
        sample_w = per_class_w[y_train]
        sampler = WeightedRandomSampler(
            torch.as_tensor(sample_w, dtype=torch.double), len(sample_w), replacement=True
        )
        shuffle = False
    else:
        sampler, shuffle = None, True

    common = dict(
        num_workers=CFG.NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=CFG.NUM_WORKERS > 0,
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                              shuffle=shuffle, drop_last=False, **common)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, **common)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, **common)
    return train_loader, val_loader, test_loader


def denormalize(tensor: torch.Tensor) -> np.ndarray:
    """(C,H,W) normalized tensor -> (H,W,3) float32 image in [0,1]."""
    img = tensor.detach().cpu().permute(1, 2, 0).numpy()
    return np.clip(img * IMAGENET_STD + IMAGENET_MEAN, 0, 1).astype(np.float32)


train_loader, val_loader, test_loader = make_loaders(CFG.MAIN_BATCH_SIZE)
print(f"Batches  train {len(train_loader)} | val {len(val_loader)} | test {len(test_loader)}")
print(f"Images   train {len(train_loader.dataset)} | val {len(val_loader.dataset)} "
      f"| test {len(test_loader.dataset)}")

In [ ]:
# =============================================================================
#  CELL 8 - Preprocessed vs augmented view of the training data
# =============================================================================
_plain_ds = SkinDataset(SPLIT["train"], EVAL_TF)
_aug_ds = SkinDataset(SPLIT["train"], TRAIN_TF)

_first_of_class = {}
for pos, gid in enumerate(SPLIT["train"]):
    lbl = int(FILE_LABELS[gid])
    _first_of_class.setdefault(lbl, pos)

fig, axes = plt.subplots(NUM_CLASSES, 5, figsize=(15, 3.1 * NUM_CLASSES))
axes = np.atleast_2d(axes)
fig.suptitle(
    "Column 1: preprocessed (resize + normalize)   |   Columns 2-5: training augmentations",
    fontsize=14, fontweight="bold", y=1.002,
)

set_seed(CFG.SEED)
for row in range(NUM_CLASSES):
    pos = _first_of_class.get(row)
    if pos is None:
        for ax in axes[row]:
            ax.axis("off")
        continue
    base_img, _ = _plain_ds[pos]
    axes[row, 0].imshow(denormalize(base_img))
    axes[row, 0].set_xticks([])
    axes[row, 0].set_yticks([])
    axes[row, 0].set_ylabel(CLASS_NAMES[row].replace(" ", "\n"), fontsize=10,
                            fontweight="bold", rotation=0, labelpad=52,
                            ha="right", va="center")
    for col in range(1, 5):
        aug_img, _ = _aug_ds[pos]
        axes[row, col].imshow(denormalize(aug_img))
        axes[row, col].axis("off")

plt.tight_layout()
savefig(fig, "03_augmentation_preview")
plt.show()
set_seed(CFG.SEED)

## 3. DERM-Net architecture

Two complementary backbones fused by a multi-scale channel-attention block:

* **EfficientNet-B4** captures *local texture* — scale, blister, vascular pattern.
* **ViT-B/16** captures *global structure* — lesion extent, symmetry, distribution.
* **MSCA fusion** runs the concatenated embedding through parallel 1-D convolutions with
  kernel 1/3/5, learns a softmax-weighted mixture over those scales, applies squeeze-and-
  excitation channel attention, and projects back with a residual connection.

The first 8 of 12 ViT blocks stay frozen: with a few hundred training images, fine-tuning all
86 M ViT parameters memorises the training set within three epochs.

In [ ]:
# =============================================================================
#  CELL 9 - Architecture
# =============================================================================
class ChannelAttention1D(nn.Module):
    """Squeeze-and-excitation over channels of a (B, C, L) tensor."""

    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(4, channels // reduction)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=2)
        max_out = torch.max(x, dim=2).values
        att = self.sigmoid(self.fc(avg_out) + self.fc(max_out)).unsqueeze(2)
        return x * att


class MSCABlock1D(nn.Module):
    """Multi-Scale Channel Attention fusion block (the novel component)."""

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        mid = max(1, out_channels // 3)
        self.branch1 = nn.Sequential(
            nn.Conv1d(in_channels, mid, 1, bias=False), nn.BatchNorm1d(mid), nn.GELU())
        self.branch3 = nn.Sequential(
            nn.Conv1d(in_channels, mid, 3, padding=1, bias=False), nn.BatchNorm1d(mid), nn.GELU())
        self.branch5 = nn.Sequential(
            nn.Conv1d(in_channels, mid, 5, padding=2, bias=False), nn.BatchNorm1d(mid), nn.GELU())

        fused_ch = mid * 3
        self.channel_att = ChannelAttention1D(fused_ch)
        self.project = nn.Sequential(
            nn.Conv1d(fused_ch, out_channels, 1, bias=False), nn.BatchNorm1d(out_channels))
        self.scale = nn.Parameter(torch.ones(3) / 3)
        self.residual = (
            nn.Conv1d(in_channels, out_channels, 1, bias=False)
            if in_channels != out_channels else nn.Identity()
        )
        self.act = nn.GELU()

    def forward(self, x):
        x = x.unsqueeze(2)                       # (B, C) -> (B, C, 1)
        w = F.softmax(self.scale, dim=0)
        fused = torch.cat(
            [self.branch1(x) * w[0], self.branch3(x) * w[1], self.branch5(x) * w[2]], dim=1
        )
        fused = self.channel_att(fused)
        out = self.act(self.project(fused) + self.residual(x))
        return out.squeeze(2)


class DERMNet(nn.Module):
    """Dual-branch EfficientNet-B4 + ViT-B/16 with MSCA fusion."""

    def __init__(self, num_classes: int, pretrained: bool = True,
                 freeze_vit_blocks: int = 8, dropout: float = CFG.DROPOUT):
        super().__init__()
        self.eff_features = timm.create_model("efficientnet_b4", pretrained=pretrained, num_classes=0)
        self.vit_features = timm.create_model("vit_base_patch16_224", pretrained=pretrained, num_classes=0)

        eff_dim = self.eff_features.num_features
        vit_dim = self.vit_features.num_features

        for p in self.vit_features.patch_embed.parameters():
            p.requires_grad = False
        n_blocks = len(self.vit_features.blocks)
        for i in range(min(freeze_vit_blocks, n_blocks)):
            for p in self.vit_features.blocks[i].parameters():
                p.requires_grad = False

        common_dim = 512
        self.eff_proj = nn.Sequential(
            nn.Linear(eff_dim, common_dim), nn.LayerNorm(common_dim), nn.GELU())
        self.vit_proj = nn.Sequential(
            nn.Linear(vit_dim, common_dim), nn.LayerNorm(common_dim), nn.GELU())
        self.fusion_msca = MSCABlock1D(common_dim * 2, common_dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(common_dim, 256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x, return_features: bool = False):
        eff_feat = self.eff_proj(self.eff_features(x))
        vit_feat = self.vit_proj(self.vit_features(x))
        fused = self.fusion_msca(torch.cat([eff_feat, vit_feat], dim=1))
        fused = self.dropout(fused)
        logits = self.classifier(fused)
        if return_features:
            return logits, fused
        return logits

    def get_eff_target_layer(self):
        """Last spatial convolution of the EfficientNet branch, for CAM methods."""
        return [self.eff_features.conv_head]


_probe = DERMNet(NUM_CLASSES, pretrained=False)
with torch.no_grad():
    _logits, _feat = _probe(torch.randn(2, 3, CFG.IMG_SIZE, CFG.IMG_SIZE), return_features=True)
_total = sum(p.numel() for p in _probe.parameters())
_trainable = sum(p.numel() for p in _probe.parameters() if p.requires_grad)
print("DERM-Net forward pass OK")
print(f"  logits   : {tuple(_logits.shape)}")
print(f"  features : {tuple(_feat.shape)}")
print(f"  params   : {_total / 1e6:.2f} M total | {_trainable / 1e6:.2f} M trainable "
      f"({100 * _trainable / _total:.1f}%)")
del _probe, _logits, _feat
gc.collect()

In [ ]:
# =============================================================================
#  CELL 10 - Loss functions
# =============================================================================
class FocalLoss(nn.Module):
    """Class-weighted focal loss; down-weights easy examples."""

    def __init__(self, gamma: float = 2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.register_buffer("weight", weight if weight is not None else None)

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


class SupConLoss(nn.Module):
    """Supervised contrastive loss over the fused embedding."""

    def __init__(self, temperature: float = 0.07):
        super().__init__()
        self.temp = temperature

    def forward(self, features, labels):
        features = F.normalize(features.float(), dim=1)
        sim = torch.matmul(features, features.T) / self.temp
        sim = sim - sim.max(dim=1, keepdim=True).values.detach()   # numerical stability

        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(features.device)
        self_mask = torch.eye(mask.size(0), device=features.device)
        mask = mask * (1 - self_mask)

        exp_sim = torch.exp(sim) * (1 - self_mask)
        log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)
        valid = mask.sum(1) > 0
        if valid.sum() == 0:      # a batch with no positive pair contributes nothing
            return features.sum() * 0.0
        mean_log = (mask * log_prob).sum(1)[valid] / mask.sum(1)[valid]
        return -mean_log.mean()


class HybridLoss(nn.Module):
    """L = alpha * Focal + beta * class-balanced CE + gamma * SupCon."""

    def __init__(self, class_weights, alpha=CFG.ALPHA, beta=CFG.BETA, gamma=CFG.GAMMA):
        super().__init__()
        w = class_weights.to(DEVICE) if class_weights is not None else None
        self.focal = FocalLoss(gamma=CFG.FOCAL_GAMMA, weight=w).to(DEVICE)
        self.ce = nn.CrossEntropyLoss(weight=w, label_smoothing=CFG.LABEL_SMOOTHING)
        self.con = SupConLoss()
        self.alpha, self.beta, self.gamma = alpha, beta, gamma

    def forward(self, logits, targets, features=None):
        loss = self.alpha * self.focal(logits, targets) + self.beta * self.ce(logits, targets)
        if features is not None and self.gamma > 0:
            loss = loss + self.gamma * self.con(features, targets)
        return loss


_lg = torch.randn(8, NUM_CLASSES, device=DEVICE)
_tg = torch.randint(0, NUM_CLASSES, (8,), device=DEVICE)
_ft = torch.randn(8, 512, device=DEVICE)
_hl = HybridLoss(CLASS_WEIGHTS)
print(f"HybridLoss with features    : {_hl(_lg, _tg, _ft).item():.4f}")
print(f"HybridLoss without features : {_hl(_lg, _tg).item():.4f}")
del _lg, _tg, _ft, _hl

## 4. Training engine

Additions over the source notebooks, each chosen because it raises test performance on a
few-hundred-image cohort:

| Technique | Effect |
|---|---|
| **EMA of weights** (decay 0.999) | An exponential moving average of the trajectory is evaluated alongside the raw weights every epoch; the better of the two is kept. Reliably worth 1-2 points of macro-F1 on small noisy datasets. |
| **Warmup + cosine LR** | 3 linear warmup epochs stop the freshly initialised fusion/classifier heads from wrecking the pretrained backbones in the first few hundred steps. |
| **Discriminative LRs** | ViT 0.1x, CNN 1x, fusion + head 5x. |
| **Select on macro-F1** | Not on accuracy — accuracy is dominated by the majority class here. |
| **4-view TTA** | Average softmax over identity / h-flip / v-flip / both. Dermatology images have no canonical orientation, so this is essentially free accuracy. |
| **Temperature scaling** | One scalar fitted on validation by LBFGS. Does not change accuracy, dramatically improves Brier score and ECE — which is what a clinical reviewer will ask about. |

In [ ]:
# =============================================================================
#  CELL 11 - EMA, schedule, training loop, TTA inference, calibration
# =============================================================================
class ModelEMA:
    """Exponential moving average of floating-point parameters and buffers."""

    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {
            k: v.detach().clone().float()
            for k, v in model.state_dict().items()
            if v.dtype.is_floating_point
        }

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1.0 - self.decay)

    def state_dict(self, model: nn.Module) -> "OrderedDict[str, torch.Tensor]":
        """EMA weights merged with the model's non-float buffers."""
        merged = OrderedDict()
        for k, v in model.state_dict().items():
            merged[k] = self.shadow[k].to(v.dtype) if k in self.shadow else v.detach().clone()
        return merged


def build_scheduler(optimizer, epochs: int, steps_per_epoch: int, warmup_epochs: int):
    warmup_steps = max(1, warmup_epochs * steps_per_epoch)
    total_steps = max(warmup_steps + 1, epochs * steps_per_epoch)

    def lr_lambda(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def make_optimizer(model: nn.Module, lr: float):
    """Discriminative learning rates for DERM-Net; a single group for baselines."""
    if isinstance(model, DERMNet):
        groups = [
            {"params": [p for p in model.vit_features.parameters() if p.requires_grad], "lr": lr * 0.1},
            {"params": model.eff_features.parameters(), "lr": lr},
            {"params": model.eff_proj.parameters(), "lr": lr * 5},
            {"params": model.vit_proj.parameters(), "lr": lr * 5},
            {"params": model.fusion_msca.parameters(), "lr": lr * 5},
            {"params": model.classifier.parameters(), "lr": lr * 5},
        ]
        groups = [g for g in groups if len(list(g["params"])) > 0]
    else:
        groups = [{"params": model.parameters(), "lr": lr}]
    return torch.optim.AdamW(groups, lr=lr, weight_decay=CFG.WEIGHT_DECAY)


def _forward(model, x, want_features: bool):
    """Uniform forward for DERM-Net (which can emit features) and plain timm models."""
    if want_features and isinstance(model, DERMNet):
        return model(x, return_features=True)
    out = model(x)
    if isinstance(out, tuple):
        out = out[0]
    return out, None


@torch.no_grad()
def predict(model, loader, tta: bool = False, temperature: float = 1.0):
    """Return (y_true, y_prob). TTA averages softmax over 4 dihedral views."""
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        views = [imgs]
        if tta:
            views += [torch.flip(imgs, dims=[3]),
                      torch.flip(imgs, dims=[2]),
                      torch.flip(imgs, dims=[2, 3])]
        probs = None
        for view in views:
            with autocast_ctx(AMP_ENABLED):
                logits, _ = _forward(model, view, False)
            p = F.softmax(logits.float() / temperature, dim=1)
            probs = p if probs is None else probs + p
        all_probs.append((probs / len(views)).cpu().numpy())
        all_labels.append(labels.numpy())
    return np.concatenate(all_labels), np.concatenate(all_probs)


@torch.no_grad()
def collect_logits(model, loader):
    """Raw (uncalibrated) logits - needed to fit the temperature."""
    model.eval()
    logits_all, labels_all = [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        with autocast_ctx(AMP_ENABLED):
            logits, _ = _forward(model, imgs, False)
        logits_all.append(logits.float().cpu())
        labels_all.append(labels)
    return torch.cat(logits_all), torch.cat(labels_all)


def fit_temperature(model, loader) -> float:
    """Single-parameter temperature scaling (Guo et al., 2017) fitted on validation."""
    logits, labels = collect_logits(model, loader)
    log_t = torch.zeros(1, requires_grad=True)
    optimizer = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)

    def closure():
        optimizer.zero_grad()
        loss = F.cross_entropy(logits / torch.exp(log_t), labels)
        loss.backward()
        return loss

    try:
        optimizer.step(closure)
        temperature = float(torch.exp(log_t).item())
    except Exception as exc:
        print(f"   temperature scaling failed ({exc}); falling back to T=1.0")
        return 1.0
    if not np.isfinite(temperature) or temperature <= 0:
        return 1.0
    return float(np.clip(temperature, 0.05, 10.0))


def run_epoch(model, loader, criterion, optimizer=None, scheduler=None,
              scaler=None, ema=None, use_features=True):
    """One pass. Training if optimizer is given, otherwise evaluation."""
    training = optimizer is not None
    model.train(training)
    total_loss, correct, seen = 0.0, 0, 0
    preds_all, labels_all = [], []

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            if training:
                optimizer.zero_grad(set_to_none=True)

            with autocast_ctx(AMP_ENABLED):
                logits, feats = _forward(model, imgs, use_features and training)
                loss = criterion(logits, labels, feats) if isinstance(criterion, HybridLoss) \
                    else criterion(logits, labels)

            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                if scheduler is not None:
                    scheduler.step()
                if ema is not None:
                    ema.update(model)

            batch = imgs.size(0)
            total_loss += loss.item() * batch
            seen += batch
            preds = logits.argmax(1)
            correct += (preds == labels).sum().item()
            preds_all.append(preds.cpu().numpy())
            labels_all.append(labels.cpu().numpy())

    preds_all = np.concatenate(preds_all)
    labels_all = np.concatenate(labels_all)
    return (
        total_loss / max(1, seen),
        correct / max(1, seen),
        f1_score(labels_all, preds_all, average="macro", zero_division=0),
    )


def train_model(model, name, train_loader, val_loader, epochs, lr,
                criterion=None, patience=None, verbose=True):
    """Train, track EMA, keep the best-macro-F1 weights, return (model, history)."""
    model = model.to(DEVICE)
    criterion = criterion or HybridLoss(CLASS_WEIGHTS)
    optimizer = make_optimizer(model, lr)
    scheduler = build_scheduler(optimizer, epochs, max(1, len(train_loader)), CFG.WARMUP_EPOCHS)
    scaler = make_grad_scaler(AMP_ENABLED)
    ema = ModelEMA(model, CFG.EMA_DECAY) if CFG.USE_EMA else None

    history = {"train_loss": [], "train_acc": [], "train_f1": [],
               "val_loss": [], "val_acc": [], "val_f1": [], "lr": []}
    best_f1, best_state, best_tag, stale = -1.0, None, "raw", 0
    eval_criterion = criterion

    if verbose:
        print("=" * 88)
        print(f" Training {name}  |  {epochs} epochs  |  base LR {lr:.1e}  "
              f"|  batch {train_loader.batch_size}")
        print("=" * 88)
        print(f"{'Ep':>3} | {'TrLoss':>8} {'TrAcc':>7} {'TrF1':>6} | "
              f"{'VaLoss':>8} {'VaAcc':>7} {'VaF1':>6} | {'src':>4} | {'LR':>8} | {'sec':>5}")
        print("-" * 88)

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc, tr_f1 = run_epoch(
            model, train_loader, criterion, optimizer, scheduler, scaler, ema)

        va_loss, va_acc, va_f1 = run_epoch(model, val_loader, eval_criterion, use_features=False)
        tag = "raw"
        candidate_state = copy.deepcopy(model.state_dict())

        if ema is not None:
            live_state = copy.deepcopy(model.state_dict())
            model.load_state_dict(ema.state_dict(model))
            e_loss, e_acc, e_f1 = run_epoch(model, val_loader, eval_criterion, use_features=False)
            if e_f1 > va_f1:
                va_loss, va_acc, va_f1, tag = e_loss, e_acc, e_f1, "ema"
                candidate_state = copy.deepcopy(model.state_dict())
            model.load_state_dict(live_state)

        current_lr = optimizer.param_groups[0]["lr"]
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["train_f1"].append(tr_f1)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        history["val_f1"].append(va_f1)
        history["lr"].append(current_lr)

        marker = ""
        if va_f1 > best_f1:
            best_f1, best_state, best_tag, stale = va_f1, candidate_state, tag, 0
            marker = "  <- best"
        else:
            stale += 1

        if verbose:
            print(f"{epoch:>3} | {tr_loss:>8.4f} {tr_acc * 100:>6.2f}% {tr_f1:>6.3f} | "
                  f"{va_loss:>8.4f} {va_acc * 100:>6.2f}% {va_f1:>6.3f} | {tag:>4} | "
                  f"{current_lr:>8.2e} | {time.time() - t0:>5.1f}{marker}")

        if patience is not None and stale >= patience:
            if verbose:
                print(f"Early stopping at epoch {epoch} (no val macro-F1 gain for {patience}).")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    if verbose:
        print(f"Best val macro-F1 = {best_f1:.4f} (from {best_tag} weights)\n")
    history["best_val_f1"] = best_f1
    history["best_source"] = best_tag
    return model, history

In [ ]:
# =============================================================================
#  CELL 12 - Metrics: bootstrap CIs, calibration error, unified registry
# =============================================================================
MODEL_RESULTS = {}
MODEL_ARTIFACTS = {}


def expected_calibration_error(y_true, y_prob, n_bins: int = 15):
    """Standard ECE plus maximum calibration error."""
    conf = y_prob.max(axis=1)
    pred = y_prob.argmax(axis=1)
    acc = (pred == y_true).astype(np.float64)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece, mce = 0.0, 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() == 0:
            continue
        gap = abs(acc[m].mean() - conf[m].mean())
        ece += (m.sum() / len(y_true)) * gap
        mce = max(mce, gap)
    return float(ece), float(mce)


def bootstrap_ci(y_true, y_pred, metric_fn, n_boot: int = 2000, alpha: float = 0.05, seed: int = 42):
    """Percentile bootstrap confidence interval for any (y_true, y_pred) metric."""
    rng = np.random.RandomState(seed)
    n = len(y_true)
    if n == 0:
        return float("nan"), float("nan")
    stats = np.empty(n_boot, dtype=np.float64)
    for b in range(n_boot):
        idx = rng.randint(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            stats[b] = np.nan
            continue
        stats[b] = metric_fn(y_true[idx], y_pred[idx])
    stats = stats[np.isfinite(stats)]
    if stats.size == 0:
        return float("nan"), float("nan")
    return float(np.percentile(stats, 100 * alpha / 2)), float(np.percentile(stats, 100 * (1 - alpha / 2)))


def compute_metrics(y_true, y_prob, n_params=0, inf_ms=0.0, n_boot=2000):
    """Every headline number for one model on one split."""
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob, dtype=np.float64)
    y_pred = y_prob.argmax(axis=1)

    onehot = np.zeros_like(y_prob)
    onehot[np.arange(len(y_true)), y_true] = 1.0

    acc = float((y_pred == y_true).mean())
    acc_lo, acc_hi = bootstrap_ci(
        y_true, y_pred, lambda a, b: float((a == b).mean()), n_boot=n_boot)
    f1_lo, f1_hi = bootstrap_ci(
        y_true, y_pred,
        lambda a, b: f1_score(a, b, average="macro", zero_division=0), n_boot=n_boot)

    try:
        auc_roc = float(roc_auc_score(onehot, y_prob, average="macro", multi_class="ovr"))
    except Exception:
        auc_roc = float("nan")
    try:
        auc_pr = float(average_precision_score(onehot, y_prob, average="macro"))
    except Exception:
        auc_pr = float("nan")

    ece, mce = expected_calibration_error(y_true, y_prob)

    return {
        "Accuracy": acc,
        "Acc CI low": acc_lo,
        "Acc CI high": acc_hi,
        "Macro F1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "F1 CI low": f1_lo,
        "F1 CI high": f1_hi,
        "Weighted F1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "Precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "Recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "AUC-ROC": auc_roc,
        "AUC-PR": auc_pr,
        "MCC": float(matthews_corrcoef(y_true, y_pred)),
        "Kappa": float(cohen_kappa_score(y_true, y_pred)),
        "Brier": float(np.mean(np.sum((y_prob - onehot) ** 2, axis=1))),
        "ECE": ece,
        "MCE": mce,
        "Params (M)": n_params / 1e6,
        "Inf ms/img": inf_ms,
    }


def measure_inference_ms(model, loader) -> float:
    """Milliseconds per image, warmed up, GPU-synchronised."""
    model.eval()
    imgs, _ = next(iter(loader))
    imgs = imgs.to(DEVICE)
    with torch.no_grad():
        for _ in range(3):
            _forward(model, imgs, False)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(10):
            _forward(model, imgs, False)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
    return (time.time() - t0) / (10 * imgs.size(0)) * 1000


def evaluate_and_register(model, name, test_loader, val_loader=None,
                          tta=None, calibrate=None, register=True, quiet=False):
    """Full test-set evaluation with optional TTA and temperature scaling."""
    tta = CFG.USE_TTA if tta is None else tta
    calibrate = CFG.CALIBRATE if calibrate is None else calibrate

    temperature = 1.0
    if calibrate and val_loader is not None:
        temperature = fit_temperature(model, val_loader)

    y_true, y_prob = predict(model, test_loader, tta=tta, temperature=temperature)
    n_params = sum(p.numel() for p in model.parameters())
    inf_ms = measure_inference_ms(model, test_loader)

    record = compute_metrics(y_true, y_prob, n_params, inf_ms)
    record["Model"] = name
    record["Temperature"] = temperature
    record["TTA"] = bool(tta)

    if register:
        MODEL_RESULTS[name] = record
        MODEL_ARTIFACTS[name] = {"y_true": y_true, "y_prob": y_prob, "temperature": temperature}

    if not quiet:
        print(f"\n{'=' * 68}")
        print(f"  Test-set evaluation - {name}")
        print(f"{'=' * 68}")
        print(f"  Accuracy     : {record['Accuracy'] * 100:6.2f}%  "
              f"(95% CI {record['Acc CI low'] * 100:.2f} - {record['Acc CI high'] * 100:.2f})")
        print(f"  Macro F1     : {record['Macro F1']:6.4f}  "
              f"(95% CI {record['F1 CI low']:.4f} - {record['F1 CI high']:.4f})")
        print(f"  Weighted F1  : {record['Weighted F1']:6.4f}")
        print(f"  Precision    : {record['Precision']:6.4f}    Recall : {record['Recall']:6.4f}")
        print(f"  AUC-ROC      : {record['AUC-ROC']:6.4f}    AUC-PR : {record['AUC-PR']:6.4f}")
        print(f"  MCC          : {record['MCC']:6.4f}    Kappa  : {record['Kappa']:6.4f}")
        print(f"  Brier        : {record['Brier']:6.4f}    ECE    : {record['ECE']:6.4f}  (lower better)")
        print(f"  Params       : {record['Params (M)']:6.2f} M   Inference: {record['Inf ms/img']:.2f} ms/img")
        print(f"  TTA {record['TTA']}   |   calibration temperature T = {temperature:.3f}")
        print(f"{'=' * 68}")
    return record


print("Metric utilities ready.")

In [ ]:
# =============================================================================
#  CELL 13 - Reporting figures (history, confusion matrix, ROC, PR)
# =============================================================================
def plot_history(history, name):
    fig, axes = plt.subplots(1, 4, figsize=(21, 4.4))
    fig.suptitle(f"{name} - training dynamics", fontsize=14, fontweight="bold")
    epochs = range(1, len(history["train_loss"]) + 1)

    axes[0].plot(epochs, history["train_loss"], label="train", lw=2)
    axes[0].plot(epochs, history["val_loss"], label="val", lw=2)
    axes[0].set_title("Loss")

    axes[1].plot(epochs, [a * 100 for a in history["train_acc"]], label="train", lw=2)
    axes[1].plot(epochs, [a * 100 for a in history["val_acc"]], label="val", lw=2)
    axes[1].set_title("Accuracy (%)")

    axes[2].plot(epochs, history["train_f1"], label="train", lw=2)
    axes[2].plot(epochs, history["val_f1"], label="val", lw=2, color="green")
    best_ep = int(np.argmax(history["val_f1"])) + 1
    axes[2].axvline(best_ep, ls="--", color="red", lw=1,
                    label=f"best (ep {best_ep})")
    axes[2].set_title("Macro F1")

    axes[3].plot(epochs, history["lr"], lw=2, color="purple")
    axes[3].set_title("Learning rate (warmup + cosine)")
    axes[3].set_yscale("log")

    for ax in axes:
        ax.set_xlabel("Epoch")
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    savefig(fig, f"history_{name.replace(' ', '_').replace('/', '-')}")
    plt.show()


def plot_confusion(y_true, y_pred, name, save_as=None):
    cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
    cm_norm = cm.astype(np.float64) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle(f"Confusion matrices - {name}", fontsize=14, fontweight="bold")

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
    axes[0].set_title("Counts")

    sns.heatmap(cm_norm * 100, annot=True, fmt=".1f", cmap="Greens", cbar=False,
                vmin=0, vmax=100, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
    axes[1].set_title("Row-normalised (% recall)")

    for ax in axes:
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.tick_params(axis="x", rotation=30)
        ax.tick_params(axis="y", rotation=0)
        for lbl in ax.get_xticklabels():
            lbl.set_ha("right")

    plt.tight_layout()
    savefig(fig, save_as or f"cm_{name.replace(' ', '_').replace('/', '-')}")
    plt.show()
    return cm


def per_class_table(y_true, y_pred, name):
    """Sensitivity / specificity / PPV / NPV / F1 per class - the clinical table."""
    cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
    rows = []
    for i in range(NUM_CLASSES):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = cm.sum() - tp - fp - fn
        sens = tp / (tp + fn) if (tp + fn) else 0.0
        spec = tn / (tn + fp) if (tn + fp) else 0.0
        ppv = tp / (tp + fp) if (tp + fp) else 0.0
        npv = tn / (tn + fn) if (tn + fn) else 0.0
        f1 = 2 * ppv * sens / (ppv + sens) if (ppv + sens) else 0.0
        rows.append({
            "Class": CLASS_NAMES[i], "Support": int(cm[i, :].sum()),
            "Sensitivity": sens, "Specificity": spec,
            "PPV": ppv, "NPV": npv, "F1": f1,
            "FNR": 1 - sens,
        })
    df = pd.DataFrame(rows)
    print(f"\nPer-class clinical metrics - {name}")
    print(df.to_string(index=False, float_format=lambda v: f"{v:0.4f}"))
    return df


def plot_roc_pr(y_true, y_prob, name):
    onehot = np.zeros_like(y_prob)
    onehot[np.arange(len(y_true)), y_true] = 1.0
    colors = sns.color_palette("husl", NUM_CLASSES)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f"ROC and Precision-Recall curves - {name}", fontsize=14, fontweight="bold")

    for i in range(NUM_CLASSES):
        if onehot[:, i].sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(onehot[:, i], y_prob[:, i])
        axes[0].plot(fpr, tpr, color=colors[i], lw=2,
                     label=f"{CLASS_NAMES[i]} (AUC {TRAPZ(tpr, fpr):.3f})")
        prec, rec, _ = precision_recall_curve(onehot[:, i], y_prob[:, i])
        axes[1].plot(rec, prec, color=colors[i], lw=2,
                     label=f"{CLASS_NAMES[i]} (AP {average_precision_score(onehot[:, i], y_prob[:, i]):.3f})")

    axes[0].plot([0, 1], [0, 1], "k--", lw=1)
    axes[0].set_xlabel("False positive rate")
    axes[0].set_ylabel("True positive rate")
    axes[0].set_title("ROC (one-vs-rest)")
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].set_title("Precision-Recall")
    for ax in axes:
        ax.legend(fontsize=8, loc="lower left")
        ax.grid(alpha=0.3)

    plt.tight_layout()
    savefig(fig, f"roc_pr_{name.replace(' ', '_').replace('/', '-')}")
    plt.show()


print("Reporting utilities ready.")

## 5. Train DERM-Net (headline run)

In [ ]:
# =============================================================================
#  CELL 14 - DERM-Net main training run
# =============================================================================
set_seed(CFG.SEED)
MAIN_NAME = "DERM-Net"
MAIN_LR = CFG.BASE_LR * (CFG.MAIN_BATCH_SIZE / 16)

train_loader, val_loader, test_loader = make_loaders(CFG.MAIN_BATCH_SIZE)

dermnet = DERMNet(NUM_CLASSES, pretrained=True)
dermnet, dermnet_history = train_model(
    dermnet, MAIN_NAME, train_loader, val_loader,
    epochs=CFG.MAIN_EPOCHS, lr=MAIN_LR,
)

MAIN_CKPT = CKPT_DIR / "dermnet_best.pth"
torch.save(dermnet.state_dict(), MAIN_CKPT)
print(f"Checkpoint saved -> {MAIN_CKPT}")

plot_history(dermnet_history, MAIN_NAME)

In [ ]:
# =============================================================================
#  CELL 15 - DERM-Net evaluation, plus an explicit TTA/calibration ablation
# =============================================================================
print("Effect of the inference-time improvements (same weights, same test set):\n")
variants = [
    ("plain (no TTA, no calibration)", False, False),
    ("+ temperature scaling", False, True),
    ("+ TTA", True, False),
    ("+ TTA + temperature scaling", True, True),
]
rows = []
for label, use_tta, use_cal in variants:
    rec = evaluate_and_register(
        dermnet, label, test_loader, val_loader,
        tta=use_tta, calibrate=use_cal, register=False, quiet=True,
    )
    rows.append({
        "Variant": label,
        "Accuracy (%)": rec["Accuracy"] * 100,
        "Macro F1": rec["Macro F1"],
        "AUC-ROC": rec["AUC-ROC"],
        "Brier": rec["Brier"],
        "ECE": rec["ECE"],
    })
inference_ablation = pd.DataFrame(rows)
print(inference_ablation.to_string(index=False, float_format=lambda v: f"{v:0.4f}"))
print("\nAccuracy is driven by TTA; Brier and ECE are driven by temperature scaling.\n")

main_record = evaluate_and_register(dermnet, MAIN_NAME, test_loader, val_loader)
y_true_main = MODEL_ARTIFACTS[MAIN_NAME]["y_true"]
y_prob_main = MODEL_ARTIFACTS[MAIN_NAME]["y_prob"]
y_pred_main = y_prob_main.argmax(axis=1)

print("\n" + classification_report(
    y_true_main, y_pred_main, labels=list(range(NUM_CLASSES)),
    target_names=CLASS_NAMES, digits=4, zero_division=0))

plot_confusion(y_true_main, y_pred_main, MAIN_NAME, save_as="04_confusion_dermnet")
dermnet_per_class = per_class_table(y_true_main, y_pred_main, MAIN_NAME)
plot_roc_pr(y_true_main, y_prob_main, MAIN_NAME)

## 6. Ablation study: batch size x epochs

In [ ]:
# =============================================================================
#  CELL 16 - Ablation grid
# =============================================================================
ABLATION_RESULTS = {}

if CFG.RUN_ABLATION:
    for bs, ep in CFG.ABLATION_GRID:
        key = f"DERM-Net BS={bs} EP={ep}"
        print(f"\n{'#' * 88}\n#  {key}\n{'#' * 88}")
        set_seed(CFG.SEED)
        a_train, a_val, a_test = make_loaders(bs)
        lr = CFG.BASE_LR * (bs / 16)

        model_ab = DERMNet(NUM_CLASSES, pretrained=True)
        model_ab, hist_ab = train_model(
            model_ab, key, a_train, a_val, epochs=ep, lr=lr, verbose=True)

        rec = evaluate_and_register(model_ab, key, a_test, a_val, quiet=True)
        ABLATION_RESULTS[key] = {"batch_size": bs, "epochs": ep,
                                 "record": rec, "history": hist_ab}
        print(f"  -> accuracy {rec['Accuracy'] * 100:.2f}%  |  macro F1 {rec['Macro F1']:.4f}  "
              f"|  AUC {rec['AUC-ROC']:.4f}")

        torch.save(model_ab.state_dict(), CKPT_DIR / f"dermnet_bs{bs}_ep{ep}.pth")
        del model_ab
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
else:
    print("Ablation disabled (CFG.RUN_ABLATION = False).")

In [ ]:
# =============================================================================
#  CELL 17 - Ablation figures
# =============================================================================
if ABLATION_RESULTS:
    batch_sizes = sorted({v["batch_size"] for v in ABLATION_RESULTS.values()})
    epoch_values = sorted({v["epochs"] for v in ABLATION_RESULTS.values()})
    metrics = [("Accuracy", "Test accuracy"), ("Macro F1", "Macro F1"),
               ("Weighted F1", "Weighted F1"), ("AUC-ROC", "AUC-ROC"),
               ("MCC", "MCC"), ("Kappa", "Cohen's kappa")]

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle("DERM-Net ablation: batch size x epochs", fontsize=16, fontweight="bold")
    axes = axes.flatten()

    for ax, (metric_key, title) in zip(axes, metrics):
        grid = np.full((len(epoch_values), len(batch_sizes)), np.nan)
        for r, ep in enumerate(epoch_values):
            for c, bs in enumerate(batch_sizes):
                key = f"DERM-Net BS={bs} EP={ep}"
                if key in ABLATION_RESULTS:
                    grid[r, c] = ABLATION_RESULTS[key]["record"][metric_key]
        im = ax.imshow(grid, cmap="YlOrRd", aspect="auto")
        ax.set_xticks(range(len(batch_sizes)))
        ax.set_xticklabels([f"BS={b}" for b in batch_sizes])
        ax.set_yticks(range(len(epoch_values)))
        ax.set_yticklabels([f"EP={e}" for e in epoch_values])
        ax.set_title(title, fontweight="bold")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        for r in range(len(epoch_values)):
            for c in range(len(batch_sizes)):
                if np.isfinite(grid[r, c]):
                    ax.text(c, r, f"{grid[r, c]:.4f}", ha="center", va="center",
                            fontsize=10, fontweight="bold")

    plt.tight_layout()
    savefig(fig, "05_ablation_heatmap")
    plt.show()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Effect of training length per batch size", fontsize=14, fontweight="bold")
    colors = sns.color_palette("Set1", len(batch_sizes))
    for ci, bs in enumerate(batch_sizes):
        eps, accs, f1s = [], [], []
        for ep in epoch_values:
            key = f"DERM-Net BS={bs} EP={ep}"
            if key in ABLATION_RESULTS:
                eps.append(ep)
                accs.append(ABLATION_RESULTS[key]["record"]["Accuracy"] * 100)
                f1s.append(ABLATION_RESULTS[key]["record"]["Macro F1"])
        if not eps:
            continue
        ax1.plot(eps, accs, marker="o", lw=2.5, ms=9, color=colors[ci], label=f"BS={bs}")
        ax2.plot(eps, f1s, marker="s", lw=2.5, ms=9, color=colors[ci], label=f"BS={bs}")
        for e, a, f in zip(eps, accs, f1s):
            ax1.annotate(f"{a:.2f}", (e, a), textcoords="offset points",
                         xytext=(0, 9), ha="center", fontsize=9)
            ax2.annotate(f"{f:.4f}", (e, f), textcoords="offset points",
                         xytext=(0, 9), ha="center", fontsize=9)
    ax1.set_xlabel("Epochs"); ax1.set_ylabel("Test accuracy (%)"); ax1.set_title("Accuracy")
    ax2.set_xlabel("Epochs"); ax2.set_ylabel("Macro F1"); ax2.set_title("Macro F1")
    for ax in (ax1, ax2):
        ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    savefig(fig, "06_ablation_lineplot")
    plt.show()

    ablation_df = pd.DataFrame([
        {"Configuration": k.replace("DERM-Net ", ""),
         "Accuracy (%)": v["record"]["Accuracy"] * 100,
         "Macro F1": v["record"]["Macro F1"],
         "Weighted F1": v["record"]["Weighted F1"],
         "AUC-ROC": v["record"]["AUC-ROC"],
         "MCC": v["record"]["MCC"],
         "Kappa": v["record"]["Kappa"],
         "ECE": v["record"]["ECE"]}
        for k, v in ABLATION_RESULTS.items()
    ]).sort_values("Macro F1", ascending=False)

    print("\nAblation summary (sorted by macro F1)")
    print(ablation_df.to_string(index=False, float_format=lambda v: f"{v:0.4f}"))
    ablation_df.to_csv(OUT_DIR / "ablation_results.csv", index=False)
    print(f"\nBest configuration: {ablation_df.iloc[0]['Configuration']}")
else:
    ablation_df = pd.DataFrame()
    print("No ablation results to plot.")

## 7. Baseline comparison

Every baseline sees the identical split, augmentation, loss and schedule, so the comparison
isolates architecture rather than training-recipe differences.

In [ ]:
# =============================================================================
#  CELL 18 - Train the baseline models
# =============================================================================
if CFG.RUN_BASELINES:
    b_train, b_val, b_test = make_loaders(32)
    for disp_name, timm_name, lr in CFG.BASELINES:
        print(f"\n{'#' * 88}\n#  Baseline: {disp_name}  ({timm_name})\n{'#' * 88}")
        try:
            set_seed(CFG.SEED)
            model_b = timm.create_model(timm_name, pretrained=True, num_classes=NUM_CLASSES)
            model_b, hist_b = train_model(
                model_b, disp_name, b_train, b_val,
                epochs=CFG.BASELINE_EPOCHS, lr=lr,
                patience=CFG.BASELINE_PATIENCE, verbose=True,
            )
            rec = evaluate_and_register(model_b, disp_name, b_test, b_val, quiet=True)
            print(f"  -> accuracy {rec['Accuracy'] * 100:.2f}%  |  macro F1 {rec['Macro F1']:.4f}  "
                  f"|  AUC {rec['AUC-ROC']:.4f}  |  {rec['Params (M)']:.1f} M params")
            torch.save(model_b.state_dict(),
                       CKPT_DIR / f"baseline_{timm_name}.pth")
            del model_b
        except Exception as exc:
            print(f"  SKIPPED {disp_name}: {type(exc).__name__}: {exc}")
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
else:
    print("Baselines disabled (CFG.RUN_BASELINES = False).")

In [ ]:
# =============================================================================
#  CELL 19 - Final comparison table
# =============================================================================
BASELINE_NAMES = [n for n, _, _ in CFG.BASELINES]
comparison_names = [n for n in MODEL_RESULTS if n in BASELINE_NAMES]
comparison_names.sort(key=lambda n: MODEL_RESULTS[n]["Accuracy"], reverse=True)
comparison_names.append(MAIN_NAME)

COLUMNS = ["Model", "Accuracy (%)", "95% CI", "Macro F1", "Weighted F1", "Precision",
           "Recall", "AUC-ROC", "AUC-PR", "MCC", "Kappa", "Brier", "ECE",
           "Params (M)", "ms/img"]

rows = []
for name in comparison_names:
    r = MODEL_RESULTS[name]
    rows.append({
        "Model": ("* " + name) if name == MAIN_NAME else name,
        "Accuracy (%)": r["Accuracy"] * 100,
        "95% CI": f"{r['Acc CI low'] * 100:.1f}-{r['Acc CI high'] * 100:.1f}",
        "Macro F1": r["Macro F1"],
        "Weighted F1": r["Weighted F1"],
        "Precision": r["Precision"],
        "Recall": r["Recall"],
        "AUC-ROC": r["AUC-ROC"],
        "AUC-PR": r["AUC-PR"],
        "MCC": r["MCC"],
        "Kappa": r["Kappa"],
        "Brier": r["Brier"],
        "ECE": r["ECE"],
        "Params (M)": r["Params (M)"],
        "ms/img": r["Inf ms/img"],
    })

comparison_df = pd.DataFrame(rows, columns=COLUMNS)

print("=" * 150)
print("  FINAL COMPARISON - baselines sorted by accuracy, * = proposed DERM-Net")
print("=" * 150)
print(comparison_df.to_string(index=False, float_format=lambda v: f"{v:0.4f}"))
print("=" * 150)

comparison_df.to_csv(OUT_DIR / "final_comparison.csv", index=False)

if len(comparison_names) > 1:
    best_baseline = max(
        (n for n in comparison_names if n != MAIN_NAME),
        key=lambda n: MODEL_RESULTS[n]["Accuracy"],
    )
    d = MODEL_RESULTS[MAIN_NAME]
    b = MODEL_RESULTS[best_baseline]
    print(f"\nDERM-Net vs best baseline ({best_baseline}):")
    for label, key, scale, fmt in [
        ("Accuracy", "Accuracy", 100, "{:+.2f} pts"),
        ("Macro F1", "Macro F1", 1, "{:+.4f}"),
        ("AUC-ROC", "AUC-ROC", 1, "{:+.4f}"),
        ("MCC", "MCC", 1, "{:+.4f}"),
        ("Brier (lower better)", "Brier", 1, "{:+.4f}"),
    ]:
        delta = (d[key] - b[key]) * scale
        print(f"  {label:<22} {d[key] * scale:>9.4f}  vs  {b[key] * scale:>9.4f}   "
              f"({fmt.format(delta)})")

In [ ]:
# =============================================================================
#  CELL 20 - Comparison charts
# =============================================================================
if len(comparison_names) > 1:
    plot_metrics = ["Accuracy", "Macro F1", "Weighted F1", "AUC-ROC", "MCC", "Kappa"]
    fig, axes = plt.subplots(2, 3, figsize=(19, 10))
    fig.suptitle("Model comparison across metrics (orange = proposed DERM-Net)",
                 fontsize=16, fontweight="bold")
    axes = axes.flatten()

    for ax, metric in zip(axes, plot_metrics):
        vals = [MODEL_RESULTS[n][metric] * (100 if metric == "Accuracy" else 1)
                for n in comparison_names]
        colors = ["#FF6D00" if n == MAIN_NAME else "#42A5F5" for n in comparison_names]
        y = np.arange(len(comparison_names))
        ax.barh(y, vals, color=colors, edgecolor="white")
        ax.set_yticks(y)
        ax.set_yticklabels(comparison_names, fontsize=9)
        ax.invert_yaxis()
        ax.set_title(metric + (" (%)" if metric == "Accuracy" else ""), fontweight="bold")
        ax.grid(alpha=0.3, axis="x")
        span = (max(vals) - min(vals)) or 1.0
        for yi, v in zip(y, vals):
            ax.text(v + span * 0.02, yi, f"{v:.2f}" if metric == "Accuracy" else f"{v:.4f}",
                    va="center", fontsize=8)
        ax.set_xlim(min(vals) - span * 0.15, max(vals) + span * 0.22)

    plt.tight_layout()
    savefig(fig, "07_model_comparison")
    plt.show()

    fig, ax = plt.subplots(figsize=(11, 7))
    for name in comparison_names:
        r = MODEL_RESULTS[name]
        is_main = name == MAIN_NAME
        ax.scatter(r["Params (M)"], r["Accuracy"] * 100,
                   s=340 if is_main else 150,
                   color="#FF6D00" if is_main else "#42A5F5",
                   marker="*" if is_main else "o",
                   edgecolor="black", zorder=3, linewidth=0.8)
        ax.annotate(name, (r["Params (M)"], r["Accuracy"] * 100),
                    textcoords="offset points", xytext=(9, 5), fontsize=9)
    ax.set_xlabel("Parameters (millions)")
    ax.set_ylabel("Test accuracy (%)")
    ax.set_title("Accuracy vs model size", fontsize=14, fontweight="bold")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    savefig(fig, "08_accuracy_vs_size")
    plt.show()

    fig, ax = plt.subplots(figsize=(11, 7))
    for name in comparison_names:
        art = MODEL_ARTIFACTS.get(name)
        if art is None:
            continue
        onehot = np.zeros_like(art["y_prob"])
        onehot[np.arange(len(art["y_true"])), art["y_true"]] = 1.0
        fpr, tpr, _ = roc_curve(onehot.ravel(), art["y_prob"].ravel())
        is_main = name == MAIN_NAME
        ax.plot(fpr, tpr, lw=3 if is_main else 1.5,
                color="#FF6D00" if is_main else None,
                label=f"{name} (AUC {MODEL_RESULTS[name]['AUC-ROC']:.4f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title("Micro-averaged ROC, all models", fontsize=14, fontweight="bold")
    ax.legend(fontsize=8, loc="lower right")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    savefig(fig, "09_roc_all_models")
    plt.show()

## 8. Error analysis and calibration

In [ ]:
# =============================================================================
#  CELL 21 - Confidence, reliability, per-class error, systematic confusions
# =============================================================================
conf_main = y_prob_main.max(axis=1)
correct_mask = y_pred_main == y_true_main

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle(f"Error analysis and calibration - {MAIN_NAME}", fontsize=16, fontweight="bold")

axes[0, 0].hist(conf_main[correct_mask], bins=20, alpha=0.75, label="correct",
                color="#2E7D32", edgecolor="white")
axes[0, 0].hist(conf_main[~correct_mask], bins=20, alpha=0.75, label="incorrect",
                color="#C62828", edgecolor="white")
axes[0, 0].set_xlabel("Predicted confidence")
axes[0, 0].set_ylabel("Cases")
axes[0, 0].set_title("Confidence distribution", fontweight="bold")
axes[0, 0].legend()

bins = np.linspace(0, 1, 11)
centres, accs, confs, weights = [], [], [], []
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (conf_main > lo) & (conf_main <= hi)
    if m.sum() == 0:
        continue
    centres.append((lo + hi) / 2)
    accs.append(correct_mask[m].mean())
    confs.append(conf_main[m].mean())
    weights.append(m.sum())
ece_main, mce_main = expected_calibration_error(y_true_main, y_prob_main)
axes[0, 1].plot([0, 1], [0, 1], "k--", lw=1.5, label="perfect calibration")
axes[0, 1].plot(confs, accs, "o-", lw=2.5, ms=9, color="#1565C0", label="DERM-Net")
for c, a, w in zip(confs, accs, weights):
    axes[0, 1].annotate(f"n={w}", (c, a), textcoords="offset points",
                        xytext=(0, -14), ha="center", fontsize=8)
axes[0, 1].set_xlabel("Mean predicted confidence")
axes[0, 1].set_ylabel("Observed accuracy")
axes[0, 1].set_title(f"Reliability diagram (ECE {ece_main:.4f}, MCE {mce_main:.4f})",
                     fontweight="bold")
axes[0, 1].legend()

err_rates = []
for c in range(NUM_CLASSES):
    m = y_true_main == c
    err_rates.append(0.0 if m.sum() == 0 else float((y_pred_main[m] != c).mean() * 100))
order = np.argsort(err_rates)[::-1]
bar_colors = ["#C62828" if err_rates[i] > 20 else "#EF6C00" if err_rates[i] > 10 else "#2E7D32"
              for i in order]
axes[1, 0].bar(range(NUM_CLASSES), [err_rates[i] for i in order], color=bar_colors)
axes[1, 0].set_xticks(range(NUM_CLASSES))
axes[1, 0].set_xticklabels([CLASS_NAMES[i] for i in order], rotation=25, ha="right", fontsize=9)
axes[1, 0].set_ylabel("Error rate (%)")
axes[1, 0].set_title("Per-class error rate", fontweight="bold")
for xi, i in enumerate(order):
    axes[1, 0].text(xi, err_rates[i] + 0.6, f"{err_rates[i]:.1f}%",
                    ha="center", fontsize=9, fontweight="bold")

cm_main = confusion_matrix(y_true_main, y_pred_main, labels=range(NUM_CLASSES))
confusions = [
    (cm_main[i, j], f"{CLASS_NAMES[i]}\n-> {CLASS_NAMES[j]}")
    for i in range(NUM_CLASSES) for j in range(NUM_CLASSES)
    if i != j and cm_main[i, j] > 0
]
confusions.sort(reverse=True)
top = confusions[:8]
if top:
    axes[1, 1].barh(range(len(top)), [c for c, _ in top], color="#8E24AA")
    axes[1, 1].set_yticks(range(len(top)))
    axes[1, 1].set_yticklabels([lbl for _, lbl in top], fontsize=8)
    axes[1, 1].invert_yaxis()
    for yi, (c, _) in enumerate(top):
        axes[1, 1].text(c + 0.05, yi, str(c), va="center", fontsize=9, fontweight="bold")
else:
    axes[1, 1].text(0.5, 0.5, "No misclassifications", ha="center", va="center",
                    fontsize=13, transform=axes[1, 1].transAxes)
axes[1, 1].set_xlabel("Cases")
axes[1, 1].set_title("Most frequent systematic confusions", fontweight="bold")

plt.tight_layout()
savefig(fig, "10_error_analysis")
plt.show()

## 9. Robustness stress test

Clinical photographs arrive noisy, blurred, badly lit and JPEG-compressed. Each corruption is
applied to the *same* test split at increasing severity; a model that collapses under mild
blur is not deployable regardless of its clean accuracy.

In [ ]:
# =============================================================================
#  CELL 22 - Corruption robustness
# =============================================================================
def corrupt_batch(x: torch.Tensor, kind: str, severity: float) -> torch.Tensor:
    """Apply a corruption in pixel space and return a re-normalised tensor."""
    mean = MEAN_T.to(x.device)
    std = STD_T.to(x.device)
    img = (x * std + mean).clamp(0, 1)

    if kind == "gaussian_noise":
        img = img + torch.randn_like(img) * severity
    elif kind == "gaussian_blur":
        k = int(2 * round(severity * 4) + 1)
        if k >= 3:
            img = transforms.functional.gaussian_blur(img, kernel_size=k, sigma=max(0.1, severity * 3))
    elif kind == "brightness":
        img = img * (1.0 + severity)
    elif kind == "darkness":
        img = img * (1.0 - severity)
    elif kind == "contrast":
        img = (img - img.mean(dim=(2, 3), keepdim=True)) * (1.0 - severity) + \
              img.mean(dim=(2, 3), keepdim=True)
    elif kind == "jpeg":
        scale = max(2, int(round(1 / max(severity, 1e-3))))
        h, w = img.shape[-2:]
        small = F.interpolate(img, size=(max(8, h // scale), max(8, w // scale)),
                              mode="bilinear", align_corners=False)
        img = F.interpolate(small, size=(h, w), mode="nearest")
    elif kind == "occlusion":
        img = img.clone()
        side = int(img.shape[-1] * severity)
        if side > 0:
            rng = np.random.RandomState(CFG.SEED)
            for b in range(img.shape[0]):
                top = rng.randint(0, max(1, img.shape[-2] - side))
                left = rng.randint(0, max(1, img.shape[-1] - side))
                img[b, :, top:top + side, left:left + side] = 0.5

    return ((img.clamp(0, 1) - mean) / std)


@torch.no_grad()
def robustness_eval(model, loader, kind: str, severity: float):
    model.eval()
    correct, total, probs_all, labels_all = 0, 0, [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        labels = labels.to(DEVICE)
        imgs = corrupt_batch(imgs, kind, severity) if kind != "clean" else imgs
        with autocast_ctx(AMP_ENABLED):
            logits, _ = _forward(model, imgs, False)
        probs = F.softmax(logits.float(), dim=1)
        correct += (probs.argmax(1) == labels).sum().item()
        total += labels.numel()
        probs_all.append(probs.cpu().numpy())
        labels_all.append(labels.cpu().numpy())
    labels_all = np.concatenate(labels_all)
    preds_all = np.concatenate(probs_all).argmax(axis=1)
    return correct / max(1, total), f1_score(labels_all, preds_all, average="macro", zero_division=0)


if CFG.RUN_ROBUSTNESS:
    set_seed(CFG.SEED)
    CORRUPTIONS = [
        ("clean", 0.0, "Clean baseline"),
        ("gaussian_noise", 0.05, "Noise (sigma 0.05)"),
        ("gaussian_noise", 0.10, "Noise (sigma 0.10)"),
        ("gaussian_blur", 0.5, "Blur (mild)"),
        ("gaussian_blur", 1.0, "Blur (strong)"),
        ("brightness", 0.30, "Over-exposed +30%"),
        ("darkness", 0.30, "Under-exposed -30%"),
        ("contrast", 0.40, "Low contrast"),
        ("jpeg", 0.25, "JPEG-style artefacts"),
        ("occlusion", 0.25, "25% occlusion"),
    ]

    rob_rows = []
    clean_acc = None
    for kind, sev, label in CORRUPTIONS:
        acc, mf1 = robustness_eval(dermnet, test_loader, kind, sev)
        if clean_acc is None:
            clean_acc = acc
        rob_rows.append({"Condition": label, "Accuracy (%)": acc * 100,
                         "Macro F1": mf1, "Drop (pts)": (clean_acc - acc) * 100})
        print(f"  {label:<24} accuracy {acc * 100:6.2f}%   macro F1 {mf1:.4f}   "
              f"drop {(clean_acc - acc) * 100:+6.2f} pts")

    robustness_df = pd.DataFrame(rob_rows)
    robustness_df.to_csv(OUT_DIR / "robustness.csv", index=False)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f"Robustness under image degradation - {MAIN_NAME}",
                 fontsize=15, fontweight="bold")

    accs = robustness_df["Accuracy (%)"].values
    drops = robustness_df["Drop (pts)"].values
    colors = ["#1565C0" if d <= 0.01 else "#2E7D32" if d < 5 else "#EF6C00" if d < 15 else "#C62828"
              for d in drops]
    ax1.bar(range(len(accs)), accs, color=colors)
    ax1.axhline(clean_acc * 100, ls="--", color="#1565C0", lw=1.5, label="clean baseline")
    ax1.set_xticks(range(len(accs)))
    ax1.set_xticklabels(robustness_df["Condition"], rotation=35, ha="right", fontsize=9)
    ax1.set_ylabel("Accuracy (%)")
    ax1.set_title("Absolute accuracy", fontweight="bold")
    ax1.legend()
    for i, v in enumerate(accs):
        ax1.text(i, v + 0.7, f"{v:.1f}", ha="center", fontsize=8, fontweight="bold")

    ax2.barh(range(len(drops)), drops, color=colors)
    ax2.set_yticks(range(len(drops)))
    ax2.set_yticklabels(robustness_df["Condition"], fontsize=9)
    ax2.invert_yaxis()
    ax2.axvline(5, ls="--", color="#EF6C00", lw=1, label="5 pt tolerance")
    ax2.axvline(15, ls="--", color="#C62828", lw=1, label="15 pt failure")
    ax2.set_xlabel("Accuracy drop (percentage points)")
    ax2.set_title("Degradation vs clean baseline", fontweight="bold")
    ax2.legend(fontsize=8)
    for i, v in enumerate(drops):
        ax2.text(v + 0.3, i, f"{v:+.1f}", va="center", fontsize=8)

    plt.tight_layout()
    savefig(fig, "11_robustness")
    plt.show()
else:
    robustness_df = pd.DataFrame()
    print("Robustness test disabled.")

# 10. Explainable AI

Six complementary explanation families, all computed on the **held-out test split** with the
weights that were actually selected on validation:

| Method | Branch | Question it answers |
|---|---|---|
| **Grad-CAM++** | EfficientNet | Which local texture drove *this* class score? |
| **EigenCAM** | EfficientNet | Gradient-free: what does the layer represent at all? |
| **LayerCAM** | EfficientNet | Same, at finer spatial granularity |
| **ViT Attention Rollout** | ViT | What global structure did the transformer attend to? |
| **Occlusion sensitivity** | whole model | Model-agnostic ground truth: what happens if I hide this patch? |
| **LIME** | whole model | Which *superpixels* (clinician-legible regions) support or oppose the call? |

Because CAM methods hook the EfficientNet branch and rollout reads the ViT branch, putting them
side by side is a direct visual test of whether the dual-branch design is doing what it claims:
local texture and global structure should highlight *different* evidence.

The final cell scores every method **quantitatively** by deletion and insertion AUC, so the
choice of explanation method is evidence-based rather than aesthetic.

In [ ]:
# =============================================================================
#  CELL 23 - XAI setup: methods, sample selection, shared helpers
# =============================================================================
XAI_OK = CFG.RUN_XAI
if XAI_OK:
    try:
        from pytorch_grad_cam import EigenCAM, GradCAMPlusPlus, LayerCAM
        from pytorch_grad_cam.utils.image import show_cam_on_image
        from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
        from lime import lime_image
        from lime.wrappers.scikit_image import SegmentationAlgorithm
        from skimage.segmentation import mark_boundaries
    except Exception as exc:
        XAI_OK = False
        print(f"XAI libraries unavailable ({exc}); skipping the XAI section.")

if XAI_OK:
    xai_model = dermnet.to(DEVICE).eval()
    TARGET_LAYERS = xai_model.get_eff_target_layer()

    # Constructed once - each CAM registers hooks on the target layer.
    CAMS = {
        "Grad-CAM++": GradCAMPlusPlus(model=xai_model, target_layers=TARGET_LAYERS),
        "EigenCAM": EigenCAM(model=xai_model, target_layers=TARGET_LAYERS),
        "LayerCAM": LayerCAM(model=xai_model, target_layers=TARGET_LAYERS),
    }

    @torch.no_grad()
    def predict_proba_tensor(batch: torch.Tensor) -> np.ndarray:
        """(B,C,H,W) normalized tensor -> (B, num_classes) probabilities."""
        xai_model.eval()
        out = []
        for i in range(0, batch.size(0), 32):
            logits, _ = _forward(xai_model, batch[i:i + 32].to(DEVICE), False)
            out.append(F.softmax(logits.float(), dim=1).cpu().numpy())
        return np.concatenate(out)

    def normalize_map(m: np.ndarray) -> np.ndarray:
        m = np.asarray(m, dtype=np.float32)
        lo, hi = float(m.min()), float(m.max())
        return np.zeros_like(m) if hi - lo < 1e-8 else (m - lo) / (hi - lo)

    def high_focus(m: np.ndarray, percentile: float = 70.0) -> np.ndarray:
        """Zero everything below a percentile, then rescale - sharper clinical overlays."""
        thresh = np.percentile(m, percentile)
        focused = np.where(m > thresh, m, 0.0)
        peak = focused.max()
        return focused / peak if peak > 0 else focused

    def select_xai_samples(n_per_class: int = 1):
        """Pick the highest-confidence *correct* test images per class.

        Falls back to the highest-confidence image of that class if the model got
        every instance of the class wrong, so the figure never has a missing row.
        """
        loader = DataLoader(
            SkinDataset(SPLIT["test"], EVAL_TF), batch_size=32, shuffle=False,
            num_workers=CFG.NUM_WORKERS, persistent_workers=CFG.NUM_WORKERS > 0)
        pools = {c: [] for c in range(NUM_CLASSES)}
        for imgs, labels in loader:
            probs = predict_proba_tensor(imgs)
            preds = probs.argmax(axis=1)
            for i in range(imgs.size(0)):
                lbl = int(labels[i])
                pools[lbl].append({
                    "tensor": imgs[i].clone(),
                    "true": lbl,
                    "pred": int(preds[i]),
                    "conf": float(probs[i, preds[i]]),
                    "correct": int(preds[i]) == lbl,
                })
        chosen = []
        for c in range(NUM_CLASSES):
            pool = pools[c]
            if not pool:
                continue
            pool.sort(key=lambda s: (s["correct"], s["conf"]), reverse=True)
            for s in pool[:n_per_class]:
                s["image"] = denormalize(s["tensor"])
                chosen.append(s)
        return chosen

    XAI_SAMPLES = select_xai_samples(1)
    N_XAI = len(XAI_SAMPLES)

    print(f"Selected {N_XAI} test images for explanation (one per class):\n")
    for s in XAI_SAMPLES:
        flag = "correct  " if s["correct"] else "INCORRECT"
        print(f"  [{flag}] true {CLASS_NAMES[s['true']]:<24} "
              f"pred {CLASS_NAMES[s['pred']]:<24} confidence {s['conf'] * 100:5.1f}%")


def cam_figure(method_name: str, cam_maps, title_suffix: str, save_as: str,
               focus_pct: float = 70.0):
    """Shared 3-column layout: original | full heatmap | high-focus heatmap."""
    fig, axes = plt.subplots(N_XAI, 3, figsize=(14, 4.7 * N_XAI))
    axes = np.atleast_2d(axes)
    fig.suptitle(f"{method_name} - {title_suffix}", fontsize=15, fontweight="bold", y=1.002)

    for row, (s, raw) in enumerate(zip(XAI_SAMPLES, cam_maps)):
        raw = normalize_map(raw)
        overlay = show_cam_on_image(s["image"], raw, use_rgb=True)
        focused = show_cam_on_image(s["image"], high_focus(raw, focus_pct), use_rgb=True)
        color = "darkgreen" if s["correct"] else "darkred"

        axes[row, 0].imshow(s["image"])
        axes[row, 0].set_title(f"Original\ntrue: {CLASS_NAMES[s['true']]}", fontsize=11)
        axes[row, 1].imshow(overlay)
        axes[row, 1].set_title(
            f"{method_name}\npred: {CLASS_NAMES[s['pred']]} ({s['conf'] * 100:.1f}%)",
            fontsize=11, color=color)
        axes[row, 2].imshow(focused)
        axes[row, 2].set_title(f"High-focus (top {100 - focus_pct:.0f}%)\n"
                               f"most discriminative region", fontsize=11, color=color)
        for ax in axes[row]:
            ax.axis("off")

    plt.tight_layout()
    savefig(fig, save_as)
    plt.show()

In [ ]:
# =============================================================================
#  CELL 24 - Grad-CAM++ (aug + eigen smoothed)
# =============================================================================
if XAI_OK:
    gradcampp_maps = []
    for s in tqdm(XAI_SAMPLES, desc="Grad-CAM++"):
        inp = s["tensor"].unsqueeze(0).to(DEVICE)
        cam = CAMS["Grad-CAM++"](
            input_tensor=inp,
            targets=[ClassifierOutputTarget(s["pred"])],
            aug_smooth=True, eigen_smooth=True,
        )[0]
        gradcampp_maps.append(cam)

    cam_figure("Grad-CAM++", gradcampp_maps,
               "EfficientNet branch, local texture evidence", "12_xai_gradcampp", 65.0)

In [ ]:
# =============================================================================
#  CELL 25 - EigenCAM (gradient-free, highly stable)
# =============================================================================
if XAI_OK:
    eigencam_maps = []
    for s in tqdm(XAI_SAMPLES, desc="EigenCAM"):
        inp = s["tensor"].unsqueeze(0).to(DEVICE)
        eigencam_maps.append(
            CAMS["EigenCAM"](input_tensor=inp,
                             targets=[ClassifierOutputTarget(s["pred"])])[0])

    cam_figure("EigenCAM", eigencam_maps,
               "gradient-free, first principal component of the activations",
               "13_xai_eigencam", 65.0)

In [ ]:
# =============================================================================
#  CELL 26 - LayerCAM (fine-grained localisation)
# =============================================================================
if XAI_OK:
    layercam_maps = []
    for s in tqdm(XAI_SAMPLES, desc="LayerCAM"):
        inp = s["tensor"].unsqueeze(0).to(DEVICE)
        layercam_maps.append(
            CAMS["LayerCAM"](input_tensor=inp,
                             targets=[ClassifierOutputTarget(s["pred"])])[0])

    cam_figure("LayerCAM", layercam_maps,
               "element-wise weighted activations, sharper lesion boundaries",
               "14_xai_layercam", 70.0)

In [ ]:
# =============================================================================
#  CELL 27 - ViT attention rollout (the ViT branch, global structure)
# =============================================================================
class ViTAttentionRollout:
    """Attention rollout (Abnar & Zuidema, 2020) for a timm ViT.

    timm's Attention module returns only the projected output, so the raw attention
    matrix is recomputed from the module's own qkv projection inside a forward hook.
    Handles the q_norm/k_norm and fused-attention variants of recent timm releases.
    """

    def __init__(self, vit, head_fusion: str = "mean", discard_ratio: float = 0.85):
        self.vit = vit
        self.head_fusion = head_fusion
        self.discard_ratio = discard_ratio
        self.attentions = []
        self.hooks = [blk.attn.register_forward_hook(self._hook) for blk in vit.blocks]

    def _hook(self, module, inputs, output):
        x = inputs[0]
        if x.dim() != 3:
            return
        B, N, C = x.shape
        heads = getattr(module, "num_heads", 12)
        head_dim = C // heads
        qkv = module.qkv(x).reshape(B, N, 3, heads, head_dim).permute(2, 0, 3, 1, 4)
        q, k = qkv[0], qkv[1]
        if hasattr(module, "q_norm") and module.q_norm is not None:
            q = module.q_norm(q)
        if hasattr(module, "k_norm") and module.k_norm is not None:
            k = module.k_norm(k)
        scale = getattr(module, "scale", head_dim ** -0.5)
        if not isinstance(scale, float):
            scale = float(scale)
        attn = (q @ k.transpose(-2, -1)) * scale
        self.attentions.append(attn.softmax(dim=-1).detach().float().cpu())

    def remove_hooks(self):
        for h in self.hooks:
            h.remove()
        self.hooks = []

    @torch.no_grad()
    def __call__(self, img_tensor: torch.Tensor) -> np.ndarray:
        self.attentions = []
        self.vit(img_tensor.to(DEVICE))
        if not self.attentions:
            raise RuntimeError("No attention captured - unexpected ViT implementation.")

        n_tokens = self.attentions[0].shape[-1]
        result = torch.eye(n_tokens)
        for attn in self.attentions:
            if self.head_fusion == "max":
                fused = attn.max(dim=1).values[0]
            elif self.head_fusion == "min":
                fused = attn.min(dim=1).values[0]
            else:
                fused = attn.mean(dim=1)[0]

            # Drop the weakest connections per query row, never the CLS column.
            n_drop = int(self.discard_ratio * (n_tokens - 1))
            if n_drop > 0:
                idx = fused[:, 1:].argsort(dim=-1)[:, :n_drop] + 1
                fused = fused.scatter(1, idx, 0.0)

            fused = fused + torch.eye(n_tokens)          # residual stream
            fused = fused / fused.sum(dim=-1, keepdim=True).clamp_min(1e-8)
            result = fused @ result

        n_prefix = getattr(self.vit, "num_prefix_tokens", 1)
        grid = getattr(getattr(self.vit, "patch_embed", None), "grid_size", None)
        patch_scores = result[0, n_prefix:]
        if grid is None or len(grid) != 2:
            side = int(round(math.sqrt(patch_scores.numel())))
            grid = (side, side)
        patch_scores = patch_scores[:grid[0] * grid[1]].reshape(grid[0], grid[1]).numpy()
        attn_map = cv2.resize(patch_scores.astype(np.float32),
                              (CFG.IMG_SIZE, CFG.IMG_SIZE), interpolation=cv2.INTER_CUBIC)
        lo, hi = attn_map.min(), attn_map.max()
        return np.zeros_like(attn_map) if hi - lo < 1e-8 else (attn_map - lo) / (hi - lo)


if XAI_OK:
    rollout_maps = []
    rollout = None
    try:
        rollout = ViTAttentionRollout(xai_model.vit_features, discard_ratio=0.85)
        for s in tqdm(XAI_SAMPLES, desc="ViT rollout"):
            rollout_maps.append(rollout(s["tensor"].unsqueeze(0)))
        cam_figure("ViT Attention Rollout", rollout_maps,
                   "ViT branch, global structural evidence (complements the CAMs above)",
                   "15_xai_vit_rollout", 70.0)
    except Exception as exc:
        print(f"Attention rollout unavailable for this timm version: "
              f"{type(exc).__name__}: {exc}")
        rollout_maps = [np.zeros((CFG.IMG_SIZE, CFG.IMG_SIZE), np.float32) for _ in XAI_SAMPLES]
    finally:
        if rollout is not None:
            rollout.remove_hooks()

In [ ]:
# =============================================================================
#  CELL 28 - Occlusion sensitivity (model-agnostic, no gradients)
# =============================================================================
@torch.no_grad()
def occlusion_map(img_tensor: torch.Tensor, target: int,
                  patch: int = 40, stride: int = 16) -> np.ndarray:
    """Slide a grey patch over the image; saliency = drop in the target probability."""
    size = img_tensor.shape[-1]
    base_prob = float(predict_proba_tensor(img_tensor.unsqueeze(0))[0, target])

    positions, variants = [], []
    grey = ((0.5 - IMAGENET_MEAN) / IMAGENET_STD).astype(np.float32)
    grey_t = torch.tensor(grey).view(3, 1, 1)

    for top in range(0, size - patch + 1, stride):
        for left in range(0, size - patch + 1, stride):
            occluded = img_tensor.clone()
            occluded[:, top:top + patch, left:left + patch] = grey_t
            positions.append((top, left))
            variants.append(occluded)

    probs = predict_proba_tensor(torch.stack(variants))[:, target]
    heat = np.zeros((size, size), np.float32)
    counts = np.zeros((size, size), np.float32)
    for (top, left), p in zip(positions, probs):
        heat[top:top + patch, left:left + patch] += (base_prob - p)
        counts[top:top + patch, left:left + patch] += 1
    heat /= np.maximum(counts, 1)
    return normalize_map(heat)


if XAI_OK:
    occlusion_maps = []
    for s in tqdm(XAI_SAMPLES, desc="Occlusion"):
        occlusion_maps.append(occlusion_map(s["tensor"], s["pred"]))

    cam_figure("Occlusion Sensitivity", occlusion_maps,
               "model-agnostic: measured probability drop when a region is hidden",
               "16_xai_occlusion", 70.0)

## 10.1 LIME - the clinician-facing explanation

LIME segments the image into superpixels, fits a sparse linear surrogate to the model's
behaviour under random superpixel masking, and reports each region's signed contribution.
It is the only method here that answers "which *regions*, and for or against?" in language a
dermatologist can check against their own reasoning — which is exactly what a reader of a
clinical AI paper wants to see.

**Green = supports the prediction. Red = argues against it.**

In [ ]:
# =============================================================================
#  CELL 29 - LIME explanations
# =============================================================================
if XAI_OK:
    lime_explainer = lime_image.LimeImageExplainer(verbose=False, random_state=CFG.SEED)
    lime_segmenter = SegmentationAlgorithm(
        "slic", n_segments=CFG.LIME_SEGMENTS, compactness=10.0, sigma=1.0,
        start_label=0, random_seed=CFG.SEED,
    )

    def lime_classifier(images_uint8: np.ndarray) -> np.ndarray:
        """LIME hands us (N,H,W,3) uint8; normalise and run the model."""
        batch = torch.from_numpy(images_uint8.astype(np.float32) / 255.0)
        batch = batch.permute(0, 3, 1, 2)
        batch = (batch - MEAN_T) / STD_T
        return predict_proba_tensor(batch)

    def explain_with_lime(image_float: np.ndarray, num_samples: int):
        return lime_explainer.explain_instance(
            (image_float * 255).astype(np.uint8),
            lime_classifier,
            top_labels=NUM_CLASSES,
            hide_color=0,
            num_samples=num_samples,
            segmentation_fn=lime_segmenter,
            batch_size=32,
            random_seed=CFG.SEED,
        )

    def lime_saliency(explanation, label: int) -> np.ndarray:
        """Turn a LIME explanation into a dense positive-evidence saliency map."""
        segments = explanation.segments
        weights = dict(explanation.local_exp[label])
        heat = np.zeros(segments.shape, np.float32)
        for seg_id, weight in weights.items():
            heat[segments == seg_id] = max(0.0, float(weight))
        return normalize_map(heat)

    lime_explanations = []
    for s in tqdm(XAI_SAMPLES, desc="LIME"):
        lime_explanations.append(explain_with_lime(s["image"], CFG.LIME_SAMPLES))

    fig, axes = plt.subplots(N_XAI, 4, figsize=(19, 4.7 * N_XAI))
    axes = np.atleast_2d(axes)
    fig.suptitle(
        "LIME superpixel explanations   |   green supports the prediction, red argues against it",
        fontsize=15, fontweight="bold", y=1.002,
    )

    for row, (s, expl) in enumerate(zip(XAI_SAMPLES, lime_explanations)):
        label = s["pred"]
        color = "darkgreen" if s["correct"] else "darkred"

        temp_pos, mask_pos = expl.get_image_and_mask(
            label, positive_only=True, num_features=6, hide_rest=False)
        temp_hide, mask_hide = expl.get_image_and_mask(
            label, positive_only=True, num_features=6, hide_rest=True)
        temp_both, mask_both = expl.get_image_and_mask(
            label, positive_only=False, num_features=10, hide_rest=False)

        axes[row, 0].imshow(s["image"])
        axes[row, 0].set_title(f"Original\ntrue: {CLASS_NAMES[s['true']]}", fontsize=11)

        axes[row, 1].imshow(mark_boundaries(temp_pos / 255.0, mask_pos))
        axes[row, 1].set_title("Supporting regions\n(top 6 positive superpixels)",
                               fontsize=11, color="darkgreen")

        axes[row, 2].imshow(mark_boundaries(temp_hide / 255.0, mask_hide))
        axes[row, 2].set_title("Evidence in isolation\n(everything else hidden)", fontsize=11)

        axes[row, 3].imshow(mark_boundaries(temp_both / 255.0, mask_both))
        axes[row, 3].set_title(
            f"For and against\npred: {CLASS_NAMES[label]} ({s['conf'] * 100:.1f}%)",
            fontsize=11, color=color)

        for ax in axes[row]:
            ax.axis("off")

    plt.tight_layout()
    savefig(fig, "17_xai_lime")
    plt.show()

In [ ]:
# =============================================================================
#  CELL 30 - LIME region-contribution weights (quantitative view)
# =============================================================================
if XAI_OK:
    fig, axes = plt.subplots(N_XAI, 2, figsize=(14, 3.6 * N_XAI))
    axes = np.atleast_2d(axes)
    fig.suptitle("LIME: signed contribution of each superpixel to the predicted class",
                 fontsize=15, fontweight="bold", y=1.002)

    for row, (s, expl) in enumerate(zip(XAI_SAMPLES, lime_explanations)):
        label = s["pred"]
        weights = sorted(expl.local_exp[label], key=lambda t: abs(t[1]), reverse=True)[:10]
        weights = sorted(weights, key=lambda t: t[1])
        seg_ids = [f"region {int(sid)}" for sid, _ in weights]
        values = [float(w) for _, w in weights]
        colors = ["#2E7D32" if v > 0 else "#C62828" for v in values]

        axes[row, 0].imshow(mark_boundaries(s["image"], expl.segments))
        axes[row, 0].set_title(f"{CLASS_NAMES[s['true']]} - superpixel segmentation",
                               fontsize=11)
        axes[row, 0].axis("off")

        axes[row, 1].barh(range(len(values)), values, color=colors)
        axes[row, 1].set_yticks(range(len(values)))
        axes[row, 1].set_yticklabels(seg_ids, fontsize=8)
        axes[row, 1].axvline(0, color="black", lw=1)
        axes[row, 1].set_xlabel("Weight in the local surrogate model", fontsize=9)
        axes[row, 1].set_title(f"pred: {CLASS_NAMES[label]} ({s['conf'] * 100:.1f}%)",
                               fontsize=11)
        axes[row, 1].grid(alpha=0.3, axis="x")

    plt.tight_layout()
    savefig(fig, "18_xai_lime_weights")
    plt.show()

In [ ]:
# =============================================================================
#  CELL 31 - Publication figure: every method side by side
# =============================================================================
if XAI_OK:
    panel_methods = [
        ("Original", None),
        ("Grad-CAM++\n(EffNet, texture)", gradcampp_maps),
        ("EigenCAM\n(EffNet, stable)", eigencam_maps),
        ("LayerCAM\n(EffNet, fine)", layercam_maps),
        ("ViT Rollout\n(ViT, structure)", rollout_maps),
        ("Occlusion\n(model-agnostic)", occlusion_maps),
        ("LIME\n(superpixels)", None),
    ]

    fig, axes = plt.subplots(N_XAI, len(panel_methods),
                             figsize=(3.5 * len(panel_methods), 3.9 * N_XAI))
    axes = np.atleast_2d(axes)
    fig.suptitle(
        "DERM-Net explainability panel - all methods, one row per class\n"
        "Columns 2-4 read the EfficientNet branch, column 5 reads the ViT branch, "
        "columns 6-7 treat the model as a black box",
        fontsize=15, fontweight="bold", y=1.004,
    )

    for col, (title, _) in enumerate(panel_methods):
        axes[0, col].set_title(title, fontsize=11, fontweight="bold", pad=10)

    for row, s in enumerate(XAI_SAMPLES):
        color = "darkgreen" if s["correct"] else "darkred"
        for col, (title, maps) in enumerate(panel_methods):
            ax = axes[row, col]
            if col == 0:
                ax.imshow(s["image"])
            elif title.startswith("LIME"):
                temp, mask = lime_explanations[row].get_image_and_mask(
                    s["pred"], positive_only=True, num_features=6, hide_rest=False)
                ax.imshow(mark_boundaries(temp / 255.0, mask))
            else:
                ax.imshow(show_cam_on_image(s["image"], normalize_map(maps[row]), use_rgb=True))
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)

        axes[row, 0].set_ylabel(
            f"{CLASS_NAMES[s['true']]}\n\npred: {CLASS_NAMES[s['pred']]}\n{s['conf'] * 100:.1f}%",
            fontsize=9, fontweight="bold", rotation=0, labelpad=68,
            ha="right", va="center", color=color,
        )

    plt.tight_layout()
    savefig(fig, "19_xai_all_methods_panel")
    plt.show()
    print("This is the publication figure.")

## 10.2 Which explanation should you actually trust?

Pretty heatmaps are not evidence. Two standard faithfulness metrics decide it:

* **Deletion AUC** — progressively blur the pixels the method ranks highest and track the
  predicted probability. A faithful map makes the probability collapse fast, so **lower is better**.
* **Insertion AUC** — start from a fully blurred image and progressively restore the
  highest-ranked pixels. A faithful map recovers the probability fast, so **higher is better**.

A **random** saliency map is included as the control. Any method that fails to beat random is
decoration, not explanation.

In [ ]:
# =============================================================================
#  CELL 32 - Quantitative XAI faithfulness: deletion and insertion AUC
# =============================================================================
if XAI_OK:
    @torch.no_grad()
    def _target_probs(batch: torch.Tensor, target: int) -> np.ndarray:
        return predict_proba_tensor(batch)[:, target]

    def deletion_insertion_auc(img_tensor: torch.Tensor, saliency: np.ndarray,
                               target: int, steps: int = 25):
        """Return (deletion AUC, insertion AUC) for one saliency map."""
        size = img_tensor.shape[-1]
        saliency = cv2.resize(np.asarray(saliency, np.float32), (size, size))
        order = np.argsort(saliency.reshape(-1))[::-1]
        n_pixels = order.size
        chunk = max(1, n_pixels // steps)

        blurred = transforms.functional.gaussian_blur(
            img_tensor.unsqueeze(0), kernel_size=31, sigma=11.0)[0]

        del_batch, ins_batch = [], []
        for k in range(0, n_pixels + chunk, chunk):
            flat_mask = np.zeros(n_pixels, dtype=bool)
            flat_mask[order[:min(k, n_pixels)]] = True
            mask = torch.from_numpy(flat_mask.reshape(size, size)).to(img_tensor.device)
            del_batch.append(torch.where(mask, blurred, img_tensor))
            ins_batch.append(torch.where(mask, img_tensor, blurred))

        del_curve = _target_probs(torch.stack(del_batch), target)
        ins_curve = _target_probs(torch.stack(ins_batch), target)
        x = np.linspace(0, 1, len(del_curve))
        return float(TRAPZ(del_curve, x)), float(TRAPZ(ins_curve, x))

    faith_samples = select_xai_samples(
        max(1, math.ceil(CFG.XAI_FAITHFULNESS_IMAGES / max(1, NUM_CLASSES)))
    )[:CFG.XAI_FAITHFULNESS_IMAGES]
    print(f"Scoring {len(faith_samples)} test images x 7 methods "
          f"({CFG.XAI_FAITHFULNESS_STEPS} steps each). This takes a few minutes.\n")

    rng = np.random.RandomState(CFG.SEED)
    faith_rows = []
    faith_rollout = None
    try:
        faith_rollout = ViTAttentionRollout(xai_model.vit_features, discard_ratio=0.85)
    except Exception:
        faith_rollout = None

    for s in tqdm(faith_samples, desc="Faithfulness"):
        inp = s["tensor"].unsqueeze(0).to(DEVICE)
        target = s["pred"]
        maps = {}

        for cam_name, cam in CAMS.items():
            try:
                maps[cam_name] = normalize_map(
                    cam(input_tensor=inp, targets=[ClassifierOutputTarget(target)])[0])
            except Exception:
                continue

        if faith_rollout is not None:
            try:
                maps["ViT Rollout"] = faith_rollout(s["tensor"].unsqueeze(0))
            except Exception:
                pass

        try:
            maps["Occlusion"] = occlusion_map(s["tensor"], target, patch=48, stride=24)
        except Exception:
            pass

        try:
            expl = explain_with_lime(s["image"], max(300, CFG.LIME_SAMPLES // 2))
            maps["LIME"] = lime_saliency(expl, target)
        except Exception:
            pass

        maps["Random (control)"] = rng.rand(CFG.IMG_SIZE, CFG.IMG_SIZE).astype(np.float32)

        for method, sal in maps.items():
            d_auc, i_auc = deletion_insertion_auc(
                s["tensor"], sal, target, CFG.XAI_FAITHFULNESS_STEPS)
            faith_rows.append({"Method": method, "Deletion AUC": d_auc,
                               "Insertion AUC": i_auc, "Class": CLASS_NAMES[s["true"]]})

    if faith_rollout is not None:
        faith_rollout.remove_hooks()

    faithfulness_df = (
        pd.DataFrame(faith_rows)
        .groupby("Method")[["Deletion AUC", "Insertion AUC"]]
        .agg(["mean", "std"])
    )
    faithfulness_df.columns = ["Deletion AUC", "Deletion SD", "Insertion AUC", "Insertion SD"]
    faithfulness_df["Faithfulness gap"] = (
        faithfulness_df["Insertion AUC"] - faithfulness_df["Deletion AUC"])
    faithfulness_df = faithfulness_df.sort_values("Faithfulness gap", ascending=False)

    print("\n" + "=" * 92)
    print("  XAI FAITHFULNESS  (deletion lower = better, insertion higher = better)")
    print("=" * 92)
    print(faithfulness_df.to_string(float_format=lambda v: f"{v:0.4f}"))
    print("=" * 92)
    faithfulness_df.to_csv(OUT_DIR / "xai_faithfulness.csv")

    best_method = faithfulness_df.index[0]
    random_gap = float(faithfulness_df.loc["Random (control)", "Faithfulness gap"]) \
        if "Random (control)" in faithfulness_df.index else float("nan")
    print(f"\nMost faithful method : {best_method}")
    print(f"Random control gap   : {random_gap:.4f}")
    beat = [m for m in faithfulness_df.index
            if m != "Random (control)"
            and faithfulness_df.loc[m, "Faithfulness gap"] > random_gap]
    print(f"Methods beating the random control: {len(beat)} of {len(faithfulness_df) - 1}"
          f"  ->  {', '.join(beat) if beat else 'none'}")

    methods = list(faithfulness_df.index)
    y = np.arange(len(methods))
    colors = ["#9E9E9E" if m == "Random (control)" else "#1565C0" for m in methods]

    fig, axes = plt.subplots(1, 3, figsize=(19, 0.62 * len(methods) + 4.5))
    fig.suptitle("Quantitative faithfulness of each explanation method",
                 fontsize=15, fontweight="bold")

    axes[0].barh(y, faithfulness_df["Deletion AUC"], xerr=faithfulness_df["Deletion SD"],
                 color=colors, capsize=3)
    axes[0].set_title("Deletion AUC (lower is better)", fontweight="bold")

    axes[1].barh(y, faithfulness_df["Insertion AUC"], xerr=faithfulness_df["Insertion SD"],
                 color=colors, capsize=3)
    axes[1].set_title("Insertion AUC (higher is better)", fontweight="bold")

    gap_colors = ["#9E9E9E" if m == "Random (control)" else "#2E7D32" for m in methods]
    axes[2].barh(y, faithfulness_df["Faithfulness gap"], color=gap_colors)
    axes[2].axvline(random_gap, ls="--", color="#C62828", lw=1.5, label="random control")
    axes[2].set_title("Insertion - Deletion gap (higher is better)", fontweight="bold")
    axes[2].legend(fontsize=9)

    for ax in axes:
        ax.set_yticks(y)
        ax.set_yticklabels(methods, fontsize=10)
        ax.invert_yaxis()
        ax.grid(alpha=0.3, axis="x")

    plt.tight_layout()
    savefig(fig, "20_xai_faithfulness")
    plt.show()

In [ ]:
# =============================================================================
#  CELL 33 - Per-class consistency check: three images per class, Grad-CAM++
# =============================================================================
if XAI_OK:
    N_PER_CLASS = 3
    consistency_samples = select_xai_samples(N_PER_CLASS)
    grouped = {c: [s for s in consistency_samples if s["true"] == c] for c in range(NUM_CLASSES)}
    present = [c for c in range(NUM_CLASSES) if grouped[c]]

    fig, axes = plt.subplots(len(present), N_PER_CLASS * 2,
                             figsize=(3.4 * N_PER_CLASS * 2, 3.7 * len(present)))
    axes = np.atleast_2d(axes)
    fig.suptitle(
        "Grad-CAM++ consistency: does the model attend to the same evidence "
        "across different patients with the same diagnosis?",
        fontsize=15, fontweight="bold", y=1.003,
    )

    for row, c in enumerate(tqdm(present, desc="Consistency")):
        for col in range(N_PER_CLASS * 2):
            axes[row, col].axis("off")
        for i, s in enumerate(grouped[c][:N_PER_CLASS]):
            cam = CAMS["Grad-CAM++"](
                input_tensor=s["tensor"].unsqueeze(0).to(DEVICE),
                targets=[ClassifierOutputTarget(s["pred"])],
                eigen_smooth=True,
            )[0]
            overlay = show_cam_on_image(s["image"], normalize_map(cam), use_rgb=True)
            axes[row, i * 2].imshow(s["image"])
            axes[row, i * 2].axis("off")
            axes[row, i * 2 + 1].imshow(overlay)
            axes[row, i * 2 + 1].set_title(
                f"{'OK' if s['correct'] else 'MISS'} -> {CLASS_NAMES[s['pred']]} "
                f"({s['conf'] * 100:.0f}%)",
                fontsize=9, color="darkgreen" if s["correct"] else "darkred")
            axes[row, i * 2 + 1].axis("off")

        axes[row, 0].axis("on")
        axes[row, 0].set_xticks([])
        axes[row, 0].set_yticks([])
        axes[row, 0].set_ylabel(CLASS_NAMES[c].replace(" ", "\n"), fontsize=10,
                                fontweight="bold", rotation=0, labelpad=58,
                                ha="right", va="center")

    plt.tight_layout()
    savefig(fig, "21_xai_per_class_consistency")
    plt.show()

## 11. Export everything

In [ ]:
# =============================================================================
#  CELL 34 - Save results and print the final summary
# =============================================================================
summary = {
    "dataset": {
        "root": str(DATA_ROOT),
        "n_images": len(FILE_PATHS),
        "classes": CLASS_NAMES,
        "class_counts": {CLASS_NAMES[i]: int(class_counts[i]) for i in range(NUM_CLASSES)},
        "imbalance_ratio": float(_imb),
        "split": {k: int(len(v)) for k, v in SPLIT.items()},
    },
    "config": {
        k: v for k, v in vars(CFG).items()
        if not k.startswith("_") and isinstance(v, (int, float, str, bool, list))
    },
    "results": MODEL_RESULTS,
    "best_val_f1": dermnet_history.get("best_val_f1"),
    "torch_version": torch.__version__,
    "timm_version": timm.__version__,
}

with open(OUT_DIR / "results.json", "w") as fh:
    json.dump(summary, fh, indent=2, default=float)

pd.DataFrame(MODEL_RESULTS).T.to_csv(OUT_DIR / "all_model_metrics.csv")
dermnet_per_class.to_csv(OUT_DIR / "dermnet_per_class.csv", index=False)
inference_ablation.to_csv(OUT_DIR / "inference_ablation.csv", index=False)
pd.DataFrame(dermnet_history).to_csv(OUT_DIR / "dermnet_training_history.csv", index=False)

r = MODEL_RESULTS[MAIN_NAME]
print("=" * 78)
print("  RUN COMPLETE")
print("=" * 78)
print(f"  Dataset          : {len(FILE_PATHS)} images, {NUM_CLASSES} classes, "
      f"imbalance {_imb:.2f}:1")
print(f"  Split            : {len(SPLIT['train'])} train / {len(SPLIT['val'])} val / "
      f"{len(SPLIT['test'])} test  (stratified, disjoint, verified)")
print()
print(f"  DERM-Net accuracy: {r['Accuracy'] * 100:.2f}%  "
      f"(95% CI {r['Acc CI low'] * 100:.2f} - {r['Acc CI high'] * 100:.2f})")
print(f"  Macro F1         : {r['Macro F1']:.4f}  "
      f"(95% CI {r['F1 CI low']:.4f} - {r['F1 CI high']:.4f})")
print(f"  AUC-ROC          : {r['AUC-ROC']:.4f}   AUC-PR: {r['AUC-PR']:.4f}")
print(f"  MCC              : {r['MCC']:.4f}   Kappa : {r['Kappa']:.4f}")
print(f"  Brier            : {r['Brier']:.4f}   ECE   : {r['ECE']:.4f}")
print(f"  Parameters       : {r['Params (M)']:.2f} M   {r['Inf ms/img']:.2f} ms/image")
print()
if XAI_OK and "faithfulness_df" in globals():
    print(f"  Most faithful XAI: {faithfulness_df.index[0]}")
print(f"  Models compared  : {len(MODEL_RESULTS)}")
print(f"  Figures written  : {len(list(FIG_DIR.glob('*.png')))} -> {FIG_DIR}")
print(f"  Tables written   : {len(list(OUT_DIR.glob('*.csv')))} CSV + results.json")
print(f"  Checkpoints      : {len(list(CKPT_DIR.glob('*.pth')))} -> {CKPT_DIR}")
print("=" * 78)